# Lesion segmentation & visualization with LINDA

A flexible, reusable workflow for:

1. Pulling or pointing at a structural MRI dataset (BIDS or flat folder).
2. Running LINDA lesion segmentation on one subject or many.
3. Visualizing lesions and atlas overlaps interactively.
4. Computing per-subject and group-level lesion statistics.

**Author of this template**: Michèle Masson-Trottier

#### Tool citation
Pustina, D., Coslett, H. B., Turkeltaub, P. E., Tustison, N., Schwartz, M. F., & Avants, B. (2016).
Automated segmentation of chronic stroke lesions using LINDA: Lesion identification with neighborhood data analysis.
*Hum Brain Mapp*, 37(4), 1405–1421. https://doi.org/10.1002/hbm.23110


## How to use this notebook

The whole workflow is driven by a single configuration cell (`CONFIG` below).
Change values there once, and every other cell will pick them up automatically.

You can run it on:

- **An OpenNeuro dataset** (default: ds004884) — fetched with `datalad`.
- **A local BIDS dataset** — point `DATASET_SOURCE` at a folder.
- **A flat folder of T1w images** — works without `sub-*/ses-*` structure.
- **A single T1w file** — for a quick one-off segmentation.

If you only want to *look* at lesions you already have, set `RUN_SEGMENTATION = False`.


## Load software tools and import python libraries

In [1]:
# Load LINDA via the Neurodesk module system
import module
await module.load('linda/0.5.1')
await module.list()


['linda/0.5.1']

## Load brain-extractor modules (required)

**Run this cell before the segmentation cell.** With the default `CONFIG["USE_HDBET_PREPROCESSING"] = True`, every LINDA run is preceded by HD-BET, so HD-BET (or SynthStrip) must be on PATH.

Set `CONFIG["USE_HDBET_PREPROCESSING"] = False` if you want the old plain-LINDA pipeline (worse masks in chronic stroke).


In [2]:
# Load HD-BET (preferred) or SynthStrip via FSL — comment out the
# one you don't want. Both can be loaded; HD-BET will be picked
# first if available.
import module
await module.load('hdbet/1.0.0')
# await module.load('fsl/6.0.7.4')   # provides mri_synthstrip
await module.list()


['linda/0.5.1', 'hdbet/1.0.0']

In [3]:
%%capture
!pip install pydicom mystmd nilearn


In [4]:
# Standard imports used throughout the notebook
import os
import re
import shutil
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nibabel as nib
from nilearn import datasets, image, plotting


## Configuration

In [5]:
# ============================================================
# CONFIGURATION — single source of truth for the whole notebook
# ============================================================
# Edit values here and re-run all cells. Every downstream cell
# reads from this CONFIG dict, so you do not need to edit them.
# ------------------------------------------------------------

CONFIG = {
    # --- Where the notebook stores its working files ----------
    # PROJECT_DIR is the root for everything the notebook creates.
    # Defaults to the folder you launched the notebook from.
    "PROJECT_DIR": Path.cwd(),

    # --- Dataset source ---------------------------------------
    # One of:
    #   "openneuro"   – fetch by accession ID with datalad
    #   "local"       – use a local BIDS dataset
    #   "flat"        – a folder of *_T1w.nii.gz files (no sub-/ses- needed)
    #   "single"      – a single T1w file
    "DATASET_SOURCE":  "openneuro",
    "OPENNEURO_ID":    "ds004884",          # used when DATASET_SOURCE == "openneuro"
    "OPENNEURO_TAG":   "1.0.2",             # git tag/branch to checkout (None = HEAD)
    "LOCAL_DATASET":   None,                # Path to local BIDS or flat folder
    "SINGLE_T1W":      None,                # Path to a single .nii.gz T1w file

    # --- Subject filter ---------------------------------------
    # Limit which subjects to process. None = process everything found.
    # You can pass a list like ["sub-M2040", "sub-M2041"], or a regex string.
    "SUBJECT_FILTER":  None,
    "SESSION_FILTER":  None,
    # If a dataset has multiple T1w runs per session, take the first by default.
    # Set to "all" to process every T1w found.
    "T1W_SELECTION":   "first",

    # --- Where outputs go -------------------------------------
    # Anatomical images are symlinked into ANAT_DIR (so we don't
    # touch the raw dataset). LINDA writes into DERIV_DIR.
    # Default matches the original notebook's folder so existing
    # runs of ds004884 are picked up automatically.
    "ANAT_SUBDIR":         "ds004884_anat",       # under PROJECT_DIR
    "DERIVATIVES_SUBDIR":  "derivatives/lesion_mask",  # under ANAT_SUBDIR

    # --- Atlas options ----------------------------------------
    "ATLAS_SUBDIR":   "atlases",                  # under PROJECT_DIR
    "ATLASES":        ["HarvardOxford", "AAL", "Destrieux", "Schaefer400"],
    "DEFAULT_ATLAS":  "HarvardOxford",            # used by single-subject views

    # --- Behaviour --------------------------------------------
    "RUN_SEGMENTATION":   True,                   # set False to only visualize
    "LESION_THRESHOLD":   0.5,                    # binarize lesion maps at this prob
    "OVERWRITE":          False,                  # re-run LINDA even if output exists

    # --- HD-BET preprocessing (default ON) -------------------
    # When True, every LINDA run is preceded by HD-BET brain
    # extraction + padding, which substantially improves the
    # final mask in chronic-stroke data (where LINDA's own
    # template-based stripper tends to over-strip and miss
    # cerebellum/brainstem). Requires HD-BET to be on PATH —
    # run the brain-extractor module-load cell near the top
    # of the notebook.
    "USE_HDBET_PREPROCESSING": True,
    "HDBET_USE_MASK_BYPASS":   True,               # pass HD-BET mask straight to LINDA's R API (skips its own brain extractor)
                                                  # the clean fix for over-stripping — leave True unless R/LINDA aren't available
    "HDBET_RSCRIPT_CMD":      "Rscript",          # path to Rscript (auto-detected if just "Rscript")
    "HDBET_PADDING_MM":        8.0,                # SKULL rim thickness around brain (medium intensity)
    "HDBET_CSF_RIM_MM":        2.0,                # CSF rim thickness inside the skull rim (dark, ~5% intensity)
    "HDBET_MODE":              "fast",             # "fast" or "accurate"
    "HDBET_DEVICE":            None,               # None=auto, "cpu", "0" (GPU)

    "FORCE_FETCH":        False,                  # re-ingest raw dataset even if ANAT_DIR is populated
    "VERBOSE":            True,
}

# ------------------------------------------------------------
# Derived paths — do not normally need to be edited
# ------------------------------------------------------------
PROJECT_DIR     = Path(CONFIG["PROJECT_DIR"]).resolve()
ANAT_DIR        = PROJECT_DIR / CONFIG["ANAT_SUBDIR"]
DERIV_DIR       = ANAT_DIR / CONFIG["DERIVATIVES_SUBDIR"]
ATLAS_DIR       = PROJECT_DIR / CONFIG["ATLAS_SUBDIR"]
RAW_DATASET_DIR = PROJECT_DIR / (CONFIG.get("OPENNEURO_ID") or "raw_dataset")

for d in (ANAT_DIR, DERIV_DIR, ATLAS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Export to the shell environment so %%bash cells can read them
os.environ["PROJECT_DIR"]     = str(PROJECT_DIR)
os.environ["ANAT_DIR"]        = str(ANAT_DIR)
os.environ["DERIV_DIR"]       = str(DERIV_DIR)
os.environ["ATLAS_DIR"]       = str(ATLAS_DIR)
os.environ["RAW_DATASET_DIR"] = str(RAW_DATASET_DIR)
os.environ["NILEARN_DATA"]    = str(ATLAS_DIR)
os.environ["DATASET_SOURCE"]  = CONFIG["DATASET_SOURCE"]
os.environ["OPENNEURO_ID"]    = CONFIG.get("OPENNEURO_ID") or ""
os.environ["OPENNEURO_TAG"]   = CONFIG.get("OPENNEURO_TAG") or ""
os.environ["LOCAL_DATASET"]   = str(CONFIG["LOCAL_DATASET"] or "")
os.environ["SINGLE_T1W"]      = str(CONFIG["SINGLE_T1W"] or "")
os.environ["FORCE_FETCH"]     = "1" if CONFIG["FORCE_FETCH"] else "0"

print("Configuration:")
for k, v in CONFIG.items():
    print(f"  {k:18s} = {v}")
print()
print("Derived paths:")
print(f"  PROJECT_DIR     = {PROJECT_DIR}")
print(f"  ANAT_DIR        = {ANAT_DIR}")
print(f"  DERIV_DIR       = {DERIV_DIR}")
print(f"  ATLAS_DIR       = {ATLAS_DIR}")
print(f"  RAW_DATASET_DIR = {RAW_DATASET_DIR}")


Configuration:
  PROJECT_DIR        = /neurodesktop-storage/aphasia-lesion-pipeline
  DATASET_SOURCE     = openneuro
  OPENNEURO_ID       = ds004884
  OPENNEURO_TAG      = 1.0.2
  LOCAL_DATASET      = None
  SINGLE_T1W         = None
  SUBJECT_FILTER     = None
  SESSION_FILTER     = None
  T1W_SELECTION      = first
  ANAT_SUBDIR        = ds004884_anat
  DERIVATIVES_SUBDIR = derivatives/lesion_mask
  ATLAS_SUBDIR       = atlases
  ATLASES            = ['HarvardOxford', 'AAL', 'Destrieux', 'Schaefer400']
  DEFAULT_ATLAS      = HarvardOxford
  RUN_SEGMENTATION   = True
  LESION_THRESHOLD   = 0.5
  OVERWRITE          = False
  USE_HDBET_PREPROCESSING = True
  HDBET_USE_MASK_BYPASS = True
  HDBET_RSCRIPT_CMD  = Rscript
  HDBET_PADDING_MM   = 8.0
  HDBET_CSF_RIM_MM   = 2.0
  HDBET_MODE         = fast
  HDBET_DEVICE       = None
  FORCE_FETCH        = False
  VERBOSE            = True

Derived paths:
  PROJECT_DIR     = /neurodesktop-storage/aphasia-lesion-pipeline
  ANAT_DIR        = /neur

### Load atlases

These atlases are used to compute regional overlap with each lesion mask.
They are cached under `ATLAS_DIR` so subsequent runs don't re-download.
Atlases are loaded into the `ALL_ATLASES` registry; downstream cells pick from it
based on `CONFIG["ATLASES"]`.


In [6]:
# ============================================================
# Atlas loader — caches everything under ATLAS_DIR
# ============================================================
print(f"Atlas directory: {ATLAS_DIR}")

class SimpleAtlas:
    """Tiny container so AAL (and any custom atlas) looks like a nilearn atlas."""
    def __init__(self, maps, labels):
        self.maps = maps
        self.labels = labels


def _fetch_harvard_oxford():
    a = datasets.fetch_atlas_harvard_oxford(
        "cort-maxprob-thr25-2mm", data_dir=str(ATLAS_DIR)
    )
    return a


def _fetch_destrieux():
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        return datasets.fetch_atlas_destrieux_2009(data_dir=str(ATLAS_DIR))


def _fetch_schaefer400():
    return datasets.fetch_atlas_schaefer_2018(
        n_rois=400, yeo_networks=7, data_dir=str(ATLAS_DIR)
    )


def _fetch_aal():
    """nilearn fetch with a fallback to a public mirror if SSL fails."""
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", DeprecationWarning)
            obj = datasets.fetch_atlas_aal(version="SPM12", data_dir=str(ATLAS_DIR))
        return SimpleAtlas(nib.load(obj.maps), obj.labels)
    except Exception as e:
        print(f"  ⚠ nilearn AAL fetch failed ({e}); using mirror")
        import requests
        import xml.etree.ElementTree as ET

        d = ATLAS_DIR / "aal_mirror"
        d.mkdir(exist_ok=True)
        nii = d / "AAL.nii.gz"
        xml = d / "AAL.xml"
        BASE = ("https://raw.githubusercontent.com/Neurita/std_brains/"
                "master/atlases/aal_SPM12/aal/atlas")

        def _dl(url, dest):
            if dest.exists():
                return
            r = requests.get(url, stream=True, timeout=60)
            r.raise_for_status()
            with open(dest, "wb") as fh:
                for chunk in r.iter_content(1024 * 1024):
                    if chunk:
                        fh.write(chunk)

        _dl(f"{BASE}/AAL.nii.gz", nii)
        _dl(f"{BASE}/AAL.xml", xml)
        labels = ["Background"]
        try:
            for elem in ET.parse(xml).getroot().iter():
                if elem.text and elem.text.strip().isalpha():
                    labels.append(elem.text.strip())
        except Exception:
            pass
        return SimpleAtlas(nib.load(str(nii)), labels)


_FETCHERS = {
    "HarvardOxford": _fetch_harvard_oxford,
    "Destrieux":     _fetch_destrieux,
    "Schaefer400":   _fetch_schaefer400,
    "AAL":           _fetch_aal,
}

ALL_ATLASES = {}
for name in CONFIG["ATLASES"]:
    if name not in _FETCHERS:
        print(f"  ⚠ Unknown atlas '{name}' — skipped")
        continue
    print(f"• Fetching {name}")
    try:
        ALL_ATLASES[name] = _FETCHERS[name]()
        n = len(ALL_ATLASES[name].labels)
        print(f"  ✓ {n} regions")
    except Exception as e:
        print(f"  ❌ Failed: {e}")

print("\nAtlases ready:", list(ALL_ATLASES.keys()))


Atlas directory: /neurodesktop-storage/aphasia-lesion-pipeline/atlases
• Fetching HarvardOxford


[fetch_atlas_harvard_oxford] Dataset found in /neurodesktop-storage/aphasia-lesion-pipeline/atlases/fsl

  ✓ 49 regions
• Fetching AAL


[fetch_atlas_aal] Dataset found in /neurodesktop-storage/aphasia-lesion-pipeline/atlases/aal_SPM12

[fetch_atlas_aal] Downloading data from https://www.gin.cnrs.fr/AAL_files/aal_for_SPM12.tar.gz ...

...

  ⚠ nilearn AAL fetch failed (HTTPSConnectionPool(host='www.gin.cnrs.fr', port=443): Max retries exceeded with url: /AAL_files/aal_for_SPM12.tar.gz (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1032)')))); using mirror
  ✓ 5 regions
• Fetching Destrieux


[fetch_atlas_destrieux_2009] Dataset found in /neurodesktop-storage/aphasia-lesion-pipeline/atlases/destrieux_2009

  ✓ 151 regions
• Fetching Schaefer400


[fetch_atlas_schaefer_2018] Dataset found in /neurodesktop-storage/aphasia-lesion-pipeline/atlases/schaefer_2018

  ✓ 401 regions

Atlases ready: ['HarvardOxford', 'AAL', 'Destrieux', 'Schaefer400']


## Data preparation

This section either fetches the dataset (OpenNeuro / datalad) or registers an
existing local dataset, and discovers the T1w images to process. Subjects/sessions
can be filtered through `CONFIG["SUBJECT_FILTER"]` and `CONFIG["SESSION_FILTER"]`.


In [7]:
%%bash
# ============================================================
# Fetch / register the source dataset
# - Skips entirely when ANAT_DIR already contains T1w files
#   (unless FORCE_FETCH=1)
# - Uses symlinks rather than copies for speed / no disk waste
# ============================================================
set -uo pipefail
shopt -s nullglob

# ---- 0. Fast path: ANAT_DIR already populated? --------------
existing=$(find "$ANAT_DIR" -maxdepth 6 -name "*_T1w.nii.gz" 2>/dev/null | wc -l)
if [ "$existing" -gt 0 ] && [ "${FORCE_FETCH:-0}" != "1" ]; then
    echo "✓ ANAT_DIR already has $existing T1w file(s) — skipping fetch."
    echo "   ANAT_DIR=$ANAT_DIR"
    echo "   (set CONFIG[\"FORCE_FETCH\"]=True in the CONFIG cell to re-fetch)"
    find "$ANAT_DIR" -maxdepth 6 -name "*_T1w.nii.gz" | head -n 20
    echo "(showing first 20)"
    exit 0
fi

case "$DATASET_SOURCE" in
  openneuro)
      if [ -z "$OPENNEURO_ID" ]; then
          echo "❌ DATASET_SOURCE=openneuro but OPENNEURO_ID is empty"; exit 1;
      fi
      if [ ! -d "$RAW_DATASET_DIR" ]; then
          echo "📥 datalad install $OPENNEURO_ID → $RAW_DATASET_DIR"
          datalad install "https://github.com/OpenNeuroDatasets/${OPENNEURO_ID}.git" \
              -p "$RAW_DATASET_DIR"
          if [ -n "$OPENNEURO_TAG" ]; then
              ( cd "$RAW_DATASET_DIR" && git checkout "$OPENNEURO_TAG" )
          fi
      else
          echo "✓ Using existing dataset at $RAW_DATASET_DIR"
      fi

      echo "📂 Linking anatomicals into $ANAT_DIR (first session per subject)"
      cd "$RAW_DATASET_DIR"
      n_subj=0; n_t1=0
      for subj in sub-*; do
          [ -d "$subj" ] || continue
          n_subj=$((n_subj + 1))
          first_ses=$(ls -d "$subj"/ses-* 2>/dev/null | head -n 1)
          if [ -n "$first_ses" ]; then
              mkdir -p "$ANAT_DIR/$first_ses/anat"
              for t1 in "$first_ses"/anat/*_T1w.nii.gz; do
                  [ -e "$t1" ] || continue
                  # fetch annex content on demand, then symlink
                  datalad get "$t1" >/dev/null 2>&1 || true
                  ln -sfn "$(realpath "$t1")" \
                      "$ANAT_DIR/$first_ses/anat/$(basename "$t1")"
                  n_t1=$((n_t1 + 1))
              done
          elif [ -d "$subj/anat" ]; then
              mkdir -p "$ANAT_DIR/$subj/anat"
              for t1 in "$subj/anat"/*_T1w.nii.gz; do
                  [ -e "$t1" ] || continue
                  datalad get "$t1" >/dev/null 2>&1 || true
                  ln -sfn "$(realpath "$t1")" \
                      "$ANAT_DIR/$subj/anat/$(basename "$t1")"
                  n_t1=$((n_t1 + 1))
              done
          fi
      done
      echo "✓ Linked $n_t1 T1w file(s) from $n_subj subject(s)"
      ;;

  local)
      if [ -z "$LOCAL_DATASET" ] || [ ! -d "$LOCAL_DATASET" ]; then
          echo "❌ LOCAL_DATASET is not a directory: $LOCAL_DATASET"; exit 1;
      fi
      echo "✓ Using local dataset at $LOCAL_DATASET"
      cd "$LOCAL_DATASET"
      n_t1=0
      for subj in sub-*; do
          [ -d "$subj" ] || continue
          for ses in "$subj"/ses-*; do
              [ -d "$ses" ] || continue
              mkdir -p "$ANAT_DIR/$ses/anat"
              for t1 in "$ses"/anat/*_T1w.nii.gz; do
                  [ -e "$t1" ] || continue
                  ln -sfn "$(realpath "$t1")" \
                      "$ANAT_DIR/$ses/anat/$(basename "$t1")"
                  n_t1=$((n_t1 + 1))
              done
          done
          if [ -d "$subj/anat" ]; then
              mkdir -p "$ANAT_DIR/$subj/anat"
              for t1 in "$subj/anat"/*_T1w.nii.gz; do
                  [ -e "$t1" ] || continue
                  ln -sfn "$(realpath "$t1")" \
                      "$ANAT_DIR/$subj/anat/$(basename "$t1")"
                  n_t1=$((n_t1 + 1))
              done
          fi
      done
      echo "✓ Linked $n_t1 T1w file(s)"
      ;;

  flat)
      if [ -z "$LOCAL_DATASET" ] || [ ! -d "$LOCAL_DATASET" ]; then
          echo "❌ LOCAL_DATASET is not a directory: $LOCAL_DATASET"; exit 1;
      fi
      echo "📂 Indexing flat folder $LOCAL_DATASET"
      mkdir -p "$ANAT_DIR/flat/anat"
      n_t1=0
      for t1 in "$LOCAL_DATASET"/*_T1w.nii.gz; do
          [ -e "$t1" ] || continue
          ln -sfn "$(realpath "$t1")" "$ANAT_DIR/flat/anat/$(basename "$t1")"
          n_t1=$((n_t1 + 1))
      done
      echo "✓ Linked $n_t1 T1w file(s)"
      ;;

  single)
      if [ -z "$SINGLE_T1W" ] || [ ! -f "$SINGLE_T1W" ]; then
          echo "❌ SINGLE_T1W is not a file: $SINGLE_T1W"; exit 1;
      fi
      mkdir -p "$ANAT_DIR/single/anat"
      ln -sfn "$(realpath "$SINGLE_T1W")" \
          "$ANAT_DIR/single/anat/$(basename "$SINGLE_T1W")"
      echo "✓ Registered 1 T1w file"
      ;;

  *)
      echo "❌ Unknown DATASET_SOURCE: $DATASET_SOURCE"; exit 1;
      ;;
esac

echo
echo "=== T1w files now visible under ANAT_DIR ==="
find "$ANAT_DIR" -maxdepth 6 -name "*_T1w.nii.gz" | head -n 20
echo "(showing first 20)"


✓ ANAT_DIR already has 10 T1w file(s) — skipping fetch.
   ANAT_DIR=/neurodesktop-storage/aphasia-lesion-pipeline/ds004884_anat
   (set CONFIG["FORCE_FETCH"]=True in the CONFIG cell to re-fetch)
/neurodesktop-storage/aphasia-lesion-pipeline/ds004884_anat/sub-M2049/ses-2530/anat/sub-M2049_ses-2530_acq-tfl3p2_run-7_T1w.nii.gz
/neurodesktop-storage/aphasia-lesion-pipeline/ds004884_anat/sub-M2040/ses-842/anat/sub-M2040_ses-842_acq-tfl3p2_run-2_T1w.nii.gz
/neurodesktop-storage/aphasia-lesion-pipeline/ds004884_anat/sub-M2047/ses-553/anat/sub-M2047_ses-553_acq-tfl3p2_run-2_T1w.nii.gz
/neurodesktop-storage/aphasia-lesion-pipeline/ds004884_anat/sub-M2046/ses-2499/anat/sub-M2046_ses-2499_acq-tfl3p2_run-2_T1w.nii.gz
/neurodesktop-storage/aphasia-lesion-pipeline/ds004884_anat/sub-M2041/ses-872/anat/sub-M2041_ses-872_acq-tfl3p2_run-3_T1w.nii.gz
/neurodesktop-storage/aphasia-lesion-pipeline/ds004884_anat/sub-M2048/ses-215/anat/sub-M2048_ses-215_acq-tfl3_run-3_T1w.nii.gz
/neurodesktop-storage/aphasia

In [8]:
# ============================================================
# Discover T1w images to process — BIDS-aware but BIDS-agnostic
# ============================================================
SUB_RE = re.compile(r"(sub-[A-Za-z0-9]+)")
SES_RE = re.compile(r"(ses-[A-Za-z0-9]+)")


def _matches_filter(value, flt):
    """Apply CONFIG filter — list of strings, single string, regex, or None."""
    if flt is None:
        return True
    if isinstance(flt, (list, tuple, set)):
        return value in flt
    if isinstance(flt, str):
        # treat as regex if it contains regex meta-chars, else equality
        if any(c in flt for c in ".*+?[](){}|^$\\"):
            return re.search(flt, value) is not None
        return value == flt
    return True


def discover_t1w(root: Path, sub_filter=None, ses_filter=None, selection="first"):
    """
    Return a list of dicts:
      {"subject": "sub-XXX", "session": "ses-YYY" or None, "t1w": Path}

    Works with:
      - sub-*/ses-*/anat/*_T1w.nii.gz   (standard BIDS w/ sessions)
      - sub-*/anat/*_T1w.nii.gz         (BIDS without sessions)
      - flat folder of *_T1w.nii.gz
    Subject/session IDs are taken from path components first,
    falling back to filename parsing.
    """
    root = Path(root)
    found = []
    for t1 in sorted(root.rglob("*_T1w.nii.gz")):
        rel = t1.relative_to(root)
        parts = rel.parts
        sub = next((p for p in parts if p.startswith("sub-")), None)
        ses = next((p for p in parts if p.startswith("ses-")), None)
        if sub is None:
            m = SUB_RE.search(t1.name)
            sub = m.group(1) if m else f"sub-{t1.stem.replace('.nii','')}"
        if ses is None:
            m = SES_RE.search(t1.name)
            ses = m.group(1) if m else None
        if not _matches_filter(sub, sub_filter):
            continue
        if ses is not None and not _matches_filter(ses, ses_filter):
            continue
        found.append({"subject": sub, "session": ses, "t1w": t1})

    if selection == "first":
        seen = {}
        for entry in found:
            key = (entry["subject"], entry["session"])
            seen.setdefault(key, entry)
        found = list(seen.values())

    return found


SUBJECTS = discover_t1w(
    ANAT_DIR,
    sub_filter=CONFIG["SUBJECT_FILTER"],
    ses_filter=CONFIG["SESSION_FILTER"],
    selection=CONFIG["T1W_SELECTION"],
)

print(f"Discovered {len(SUBJECTS)} T1w image(s) for processing.")
for s in SUBJECTS[:10]:
    print(f"  • {s['subject']}  {s['session'] or '(no session)':<12s}  {s['t1w'].name}")
if len(SUBJECTS) > 10:
    print(f"  … and {len(SUBJECTS) - 10} more")


def deriv_path_for(entry):
    """Where LINDA outputs for one entry should live (BIDS-derivatives style)."""
    sub = entry["subject"]
    ses = entry["session"]
    if ses:
        return DERIV_DIR / sub / ses / "anat" / "linda"
    return DERIV_DIR / sub / "anat" / "linda"


def lesion_mni_for(entry):
    """Convenience: where Lesion_in_MNI.nii.gz lives for an entry."""
    return deriv_path_for(entry) / "Lesion_in_MNI.nii.gz"


Discovered 10 T1w image(s) for processing.
  • sub-M2040  ses-842       sub-M2040_ses-842_acq-tfl3p2_run-2_T1w.nii.gz
  • sub-M2041  ses-872       sub-M2041_ses-872_acq-tfl3p2_run-3_T1w.nii.gz
  • sub-M2042  ses-1953      sub-M2042_ses-1953_acq-tfl3p2_run-7_T1w.nii.gz
  • sub-M2043  ses-183       sub-M2043_ses-183_acq-tfl3p2_run-3_T1w.nii.gz
  • sub-M2044  ses-3122      sub-M2044_ses-3122_acq-tfl3p2_run-2_T1w.nii.gz
  • sub-M2045  ses-381       sub-M2045_ses-381_acq-tfl3p2_run-2_T1w.nii.gz
  • sub-M2046  ses-2499      sub-M2046_ses-2499_acq-tfl3p2_run-2_T1w.nii.gz
  • sub-M2047  ses-553       sub-M2047_ses-553_acq-tfl3p2_run-2_T1w.nii.gz
  • sub-M2048  ses-215       sub-M2048_ses-215_acq-tfl3_run-3_T1w.nii.gz
  • sub-M2049  ses-2530      sub-M2049_ses-2530_acq-tfl3p2_run-7_T1w.nii.gz


## Lesion segmentation

LINDA writes its outputs into a per-subject folder under `DERIV_DIR`. Existing
outputs are skipped unless `CONFIG["OVERWRITE"]` is `True`. Set
`CONFIG["RUN_SEGMENTATION"] = False` to skip this section entirely (e.g. when you
already have lesion masks and just want to visualize).


### Test pipeline on ONE subject (recommended before batch)

Before kicking off `linda_predict.sh` on every subject in your dataset,
run this cell on a single subject to verify the HD-BET + fake-skull +
LINDA pipeline produces a reasonable brain mask and lesion. The cell
shows the result inline (no need to leave the notebook).

If it looks good, run the main batch cell below — it'll skip this
subject (already done) and process the rest. If it looks bad, change
`HDBET_MODE = "accurate"` in the CONFIG cell, set
`CONFIG["OVERWRITE"] = True`, and re-run this test cell to retry on the
same subject.


In [9]:
# ============================================================
# Test pipeline on ONE subject (recommended before batch)
# ------------------------------------------------------------
# Runs HD-BET + padding + LINDA on the single subject indexed by
# TEST_SUBJECT_IDX (see the SUBJECTS list printed by the discovery
# cell above). Outputs land in the canonical linda/ folder, so the
# main batch cell will skip this subject on its next run.
# ============================================================
TEST_SUBJECT_IDX = 2          # which subject to test (0 = first in SUBJECTS)
RUN_TEST         = True       # set False to skip this cell entirely
# ----------------------------------------------------------------

import sys, importlib
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import linda_qc as q
importlib.reload(q)

from ipyniivue import NiiVue
from IPython.display import display, HTML

if not RUN_TEST:
    print("RUN_TEST = False — skipping single-subject test.")
elif not 0 <= TEST_SUBJECT_IDX < len(SUBJECTS):
    print(f"TEST_SUBJECT_IDX={TEST_SUBJECT_IDX} out of range "
          f"(0..{len(SUBJECTS)-1}).")
else:
    use_hdbet = bool(CONFIG.get("USE_HDBET_PREPROCESSING", True))

    if use_hdbet and q.detect_brain_extractor() is None:
        print("❌ HD-BET / SynthStrip not on PATH.")
        print(q.brain_extractor_help_text())
    else:
        e = SUBJECTS[TEST_SUBJECT_IDX]
        out_dir = deriv_path_for(e)
        out_dir.mkdir(parents=True, exist_ok=True)
        expected = out_dir / "Lesion_in_MNI.nii.gz"

        tag = f"{e['subject']} {e['session'] or ''}".strip()
        print(f"Test subject: [{TEST_SUBJECT_IDX}] {tag}")
        print(f"  T1w:    {e['t1w']}")
        print(f"  Output: {out_dir}")
        print()

        if expected.exists() and expected.stat().st_size > 0 \
                and not CONFIG.get("OVERWRITE", False):
            print(f"ℹ Subject already has output — skipping (existing "
                  f"Lesion_in_MNI.nii.gz).")
            print(f"  To redo, set CONFIG['OVERWRITE'] = True in the CONFIG cell.")
        else:
            if use_hdbet:
                print(f"  Pipeline: HD-BET ({CONFIG.get('HDBET_MODE','fast')} "
                      f"+ {CONFIG.get('HDBET_PADDING_MM',8.0)}mm padding) → LINDA")
                res = q.hdbet_then_linda(
                    e["t1w"], out_dir,
                    use_mask_bypass=bool(CONFIG.get("HDBET_USE_MASK_BYPASS", True)),
                    use_padding=True,
                    padding_mm=float(CONFIG.get("HDBET_PADDING_MM", 8.0)),
                    csf_rim_mm=float(CONFIG.get("HDBET_CSF_RIM_MM", 2.0)),
                    hdbet_mode=str(CONFIG.get("HDBET_MODE", "fast")),
                    hdbet_device=CONFIG.get("HDBET_DEVICE"),
                    rscript_cmd=str(CONFIG.get("HDBET_RSCRIPT_CMD", "Rscript")),
                )
                print()
                print(f"  brain extractor rc={res.get('returncode')}, "
                      f"LINDA rc={res.get('linda_returncode')}")
                print(f"  moved {len(res.get('moved') or [])} file(s) "
                      f"to canonical location")
            else:
                import subprocess
                print(f"  Pipeline: plain LINDA (USE_HDBET_PREPROCESSING=False)")
                rc = subprocess.run(
                    ["linda_predict.sh", str(e["t1w"]), str(out_dir)],
                    check=False,
                ).returncode
                print(f"  LINDA rc={rc}")

            print()
            if expected.exists() and expected.stat().st_size > 0:
                display(HTML("<hr><b>Result preview</b>"))

                display(HTML("<small style='color:#aaa'>"
                             "N4-corrected T1 with new BrainMask overlay (red):"
                             "</small>"))
                nv1 = NiiVue()
                vols1 = []
                if (out_dir / "N4corrected.nii.gz").exists():
                    vols1.append({"path": str(out_dir / "N4corrected.nii.gz")})
                if (out_dir / "BrainMask.nii.gz").exists():
                    vols1.append({"path": str(out_dir / "BrainMask.nii.gz"),
                                  "colormap": "red", "opacity": 0.35})
                nv1.load_volumes(vols1)
                display(nv1)

                display(HTML("<small style='color:#aaa'>"
                             "Subject_in_MNI with Lesion_in_MNI overlay (red):"
                             "</small>"))
                nv2 = NiiVue()
                vols2 = []
                if (out_dir / "Subject_in_MNI.nii.gz").exists():
                    vols2.append({"path": str(out_dir / "Subject_in_MNI.nii.gz")})
                if (out_dir / "Lesion_in_MNI.nii.gz").exists():
                    vols2.append({"path": str(out_dir / "Lesion_in_MNI.nii.gz"),
                                  "colormap": "red", "opacity": 0.6})
                nv2.load_volumes(vols2)
                display(nv2)

                print()
                print("If this looks right:")
                print("  → Run the main batch cell below. It will skip this")
                print("    subject (already done) and process the rest.")
                print()
                print("If the brain mask is too tight / too loose:")
                print("  → Change CONFIG['HDBET_MODE'] to 'accurate' "
                      "(slower, better coverage)")
                print("  → Or adjust CONFIG['HDBET_PADDING_MM']")
                print("  → Set CONFIG['OVERWRITE'] = True and re-run THIS cell")
                print("    (only this subject re-runs).")
            else:
                print("⚠ Pipeline finished but no Lesion_in_MNI.nii.gz produced.")
                print("  Check the LINDA output above for errors.")


Test subject: [2] sub-M2042 ses-1953
  T1w:    /neurodesktop-storage/aphasia-lesion-pipeline/ds004884_anat/sub-M2042/ses-1953/anat/sub-M2042_ses-1953_acq-tfl3p2_run-7_T1w.nii.gz
  Output: /neurodesktop-storage/aphasia-lesion-pipeline/ds004884_anat/derivatives/lesion_mask/sub-M2042/ses-1953/anat/linda

  Pipeline: HD-BET (fast + 8.0mm padding) → LINDA
  ▶ hd-bet -i /neurodesktop-storage/aphasia-lesion-pipeline/ds004884_anat/sub-M2042/ses-1953/anat/sub-M2042_ses-1953_acq-tfl3p2_run-7_T1w.nii.gz -o /neurodesktop-storage/aphasia-lesion-pipeline/ds004884_anat/derivatives/lesion_mask/sub-M2042/ses-1953/anat/_hdbet_work/sub-M2042_ses-1953_acq-tfl3p2_run-7_T1w_hdbet -device cpu -mode fast -tta 0


/opt/HD-BET/HD_BET/utils.py:74: UserWarning: The argument 'neighbors' is deprecated and will be removed in scikit-image 0.18, use 'connectivity' instead. For neighbors=8, use connectivity=3
  lbls = label(mask, 8)



########################
If you are using hd-bet, please cite the following paper:
Isensee F, Schell M, Tursunova I, Brugnara G, Bonekamp D, Neuberger U, Wick A, Schlemmer HP, Heiland S, Wick W,Bendszus M, Maier-Hein KH, Kickingereder P. Automated brain extraction of multi-sequence MRI using artificialneural networks. arXiv preprint arXiv:1901.11341, 2019.
########################

File: /neurodesktop-storage/aphasia-lesion-pipeline/ds004884_anat/sub-M2042/ses-1953/anat/sub-M2042_ses-1953_acq-tfl3p2_run-7_T1w.nii.gz
preprocessing...
image shape after preprocessing:  (171, 171, 107)
prediction (CNN id)...
0
running postprocessing... 
exporting segmentation...
  ▶ (local bash wrapper → singularity exec) bash /neurodesktop-storage/aphasia-lesion-pipeline/linda_predict_with_mask.sh --t1 /neurodesktop-storage/aphasia-lesion-pipeline/ds004884_anat/sub-M2042/ses-1953/anat/sub-M2042_ses-1953_acq-tfl3p2_run-7_T1w.nii.gz --mask /neurodesktop-storage/aphasia-lesion-pipeline/ds004884_anat/derivat

ERROR: LINDA container not found at: '/cvmfs/neurodesk.ardc.edu.au/containers/linda_0.5.1_20250814/linda_0.5.1_20250814.simg'
  Set LINDA_SIMG to the absolute path of the LINDA .simg / .sif file.


In [10]:
# ============================================================
# Run LINDA with brain_mask= bypass via R (Neurodesk workaround)
# ------------------------------------------------------------
# The LINDA Neurodesk container deploys `R` (not `Rscript`) on the
# host PATH. This cell calls LINDA's R API directly via `R --no-save
# -e ...`, passing the HD-BET mask the test cell above just wrote.
# When LINDA gets a `brain_mask=` argument, its internal
# n4_skull_strip is bypassed entirely — the mask we hand it is used
# verbatim, which fixes the over-stripping you see on lesioned
# brains.
#
# Once the Neurodesk container is rebuilt with the
# linda_predict_with_mask.sh wrapper baked in, this cell becomes
# unnecessary — the test cell above will route through the wrapper
# directly. Until then, run THIS cell after the test cell.
# ============================================================
import importlib, linda_qc as q; importlib.reload(q)
from pathlib import Path

if not RUN_TEST or not 0 <= TEST_SUBJECT_IDX < len(SUBJECTS):
    print("Skipping R bypass cell — RUN_TEST is False or TEST_SUBJECT_IDX out of range.")
else:
    e = SUBJECTS[TEST_SUBJECT_IDX]
    out_dir = deriv_path_for(e)
    work    = out_dir.parent / "_hdbet_work"
    t1      = Path(e["t1w"])
    stem    = t1.name.replace(".nii.gz", "").replace(".nii", "")
    mask    = work / f"{stem}_brain_mask.nii.gz"

    print(f"Subject : [{TEST_SUBJECT_IDX}] {e['subject']} {e.get('session') or ''}".rstrip())
    print(f"  T1      = {t1}")
    print(f"  mask    = {mask}")
    print(f"  outdir  = {out_dir}")

    if not mask.exists():
        print()
        print(f"❌ HD-BET mask not found at {mask}")
        print("  Run the test cell above first — it produces this mask.")
    else:
        # Clear stale LINDA outputs so the bypass run isn't a cache no-op.
        cleared = 0
        for f in sorted(out_dir.glob("*.nii.gz")):
            f.unlink(); cleared += 1
        if cleared:
            print(f"  cleared {cleared} stale LINDA .nii.gz file(s) before re-run")
        print()

        rc = q.run_linda_with_mask_via_R(t1, mask, out_dir,
                                          reviewer=globals().get("LAST_REVIEWER"),
                                          cache=False)
        print(f"\nLINDA (via R, mask-bypass) rc = {rc}")

        if rc == 0:
            new_mask = out_dir / "BrainMask.nii.gz"
            if new_mask.exists():
                import nibabel as nib
                a = (nib.load(str(mask)).get_fdata() > 0).sum()
                b = (nib.load(str(new_mask)).get_fdata() > 0).sum()
                print(f"\n  HD-BET mask voxels  : {a:,}")
                print(f"  LINDA's BrainMask   : {b:,}")
                print(f"  ratio (LINDA/HD-BET): {b/a:.3f}")
                if abs(b - a) <= max(50, 0.001 * a):
                    print("  ✓ LINDA mask matches HD-BET mask — bypass is working!")
                else:
                    print("  ⚠ LINDA mask differs from HD-BET mask. "
                          "If the ratio is ~1.0 this is fine (rounding); "
                          "if it's much smaller, the bypass may not have engaged "
                          "(check the R output above for 'Brain mask provided by user').")


Subject : [2] sub-M2042 ses-1953
  T1      = /neurodesktop-storage/aphasia-lesion-pipeline/ds004884_anat/sub-M2042/ses-1953/anat/sub-M2042_ses-1953_acq-tfl3p2_run-7_T1w.nii.gz
  mask    = /neurodesktop-storage/aphasia-lesion-pipeline/ds004884_anat/derivatives/lesion_mask/sub-M2042/ses-1953/anat/_hdbet_work/sub-M2042_ses-1953_acq-tfl3p2_run-7_T1w_brain_mask.nii.gz
  outdir  = /neurodesktop-storage/aphasia-lesion-pipeline/ds004884_anat/derivatives/lesion_mask/sub-M2042/ses-1953/anat/linda

  ▶ (R --no-save) bypassing LINDA's skull-strip with HD-BET mask
      T1     = /neurodesktop-storage/aphasia-lesion-pipeline/ds004884_anat/sub-M2042/ses-1953/anat/sub-M2042_ses-1953_acq-tfl3p2_run-7_T1w.nii.gz
      mask   = /neurodesktop-storage/aphasia-lesion-pipeline/ds004884_anat/derivatives/lesion_mask/sub-M2042/ses-1953/anat/_hdbet_work/sub-M2042_ses-1953_acq-tfl3p2_run-7_T1w_brain_mask.nii.gz
      outdir = /neurodesktop-storage/aphasia-lesion-pipeline/ds004884_anat/derivatives/lesion_mask/sub-

In [12]:
# ============================================================
# Pipeline-stage diagnostic viewer
# ------------------------------------------------------------
# Visualises, for one subject:
#   1. Original T1 with HD-BET brain mask overlay
#   2. LINDA's N4-corrected brain with LINDA's BrainMask overlay
#   3. HD-BET vs LINDA mask comparison on the original T1
#   4. Final lesion mask (native space) on the original T1
#   5. Lesion in MNI space on the MNI-warped subject
# Plus a voxel-count table so you can see WHERE coverage is lost.
# ============================================================
DIAG_SUBJECT_IDX = 2     # which subject (matches the SUBJECTS index)

import sys, importlib
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import linda_qc as q
importlib.reload(q)

import nibabel as nib
from ipyniivue import NiiVue
from IPython.display import display, HTML

if not 0 <= DIAG_SUBJECT_IDX < len(SUBJECTS):
    print(f"DIAG_SUBJECT_IDX={DIAG_SUBJECT_IDX} out of range "
          f"(0..{len(SUBJECTS)-1}).")
else:
    e = SUBJECTS[DIAG_SUBJECT_IDX]
    out_dir  = deriv_path_for(e)
    work_dir = out_dir.parent / "_hdbet_work"
    t1_orig  = Path(e["t1w"])
    stem     = t1_orig.name.replace(".nii.gz", "").replace(".nii", "")

    hdbet_brain      = work_dir / f"{stem}_brain.nii.gz"
    hdbet_mask       = work_dir / f"{stem}_brain_mask.nii.gz"
    linda_n4_brain   = out_dir / "N4corrected_Brain.nii.gz"
    linda_brainmask  = out_dir / "BrainMask.nii.gz"
    lesion_native    = out_dir / "Prediction3_native.nii.gz"
    lesion_mni       = out_dir / "Lesion_in_MNI.nii.gz"
    subject_mni      = out_dir / "Subject_in_MNI.nii.gz"

    display(HTML(
        f"<h3>{e['subject']} {e['session'] or ''} "
        f"&nbsp;·&nbsp; pipeline-stage diagnostic</h3>"
    ))

    # Voxel count table
    def _vox(p):
        if not p.exists(): return None
        try:    return int((nib.load(str(p)).get_fdata() > 0).sum())
        except: return None

    rows = []
    for label, path in [
        ("Original T1",                       t1_orig),
        ("HD-BET brain (T1, skull-stripped)", hdbet_brain),
        ("HD-BET brain mask",                 hdbet_mask),
        ("LINDA N4_corrected_Brain",          linda_n4_brain),
        ("LINDA BrainMask",                   linda_brainmask),
        ("LINDA lesion (native space)",       lesion_native),
        ("LINDA lesion (MNI space)",          lesion_mni),
        ("LINDA Subject in MNI",              subject_mni),
    ]:
        n = _vox(path)
        n_str = f"{n:,}" if n is not None else "—"
        exists = "✓" if path.exists() else "✗"
        rows.append(
            f"<tr><td>{exists}</td><td>{label}</td>"
            f"<td style='text-align:right'>{n_str}</td>"
            f"<td style='font-family:monospace; font-size:11px; color:#888'>"
            f"{path}</td></tr>"
        )
    display(HTML(
        "<table style='border-collapse:collapse; font-size:13px'>"
        "<tr style='background:#2c3e50; color:white'>"
        "<th style='padding:4px 8px'></th>"
        "<th style='padding:4px 8px; text-align:left'>Stage</th>"
        "<th style='padding:4px 8px'>Voxels &gt; 0</th>"
        "<th style='padding:4px 8px; text-align:left'>Path</th></tr>"
        + "".join(rows) + "</table>"
    ))

    # Quick coverage delta if both masks exist
    if hdbet_mask.exists() and linda_brainmask.exists():
        m_h = nib.load(str(hdbet_mask)).get_fdata() > 0
        m_l = nib.load(str(linda_brainmask)).get_fdata() > 0
        kept    = int((m_h & m_l).sum())
        dropped = int((m_h & ~m_l).sum())
        added   = int((~m_h & m_l).sum())
        display(HTML(
            f"<small><b>HD-BET vs LINDA mask:</b> "
            f"both agree on {kept:,} voxels; LINDA dropped "
            f"<b style='color:#d9534f'>{dropped:,}</b> voxels HD-BET kept "
            f"({100*dropped/max(int(m_h.sum()),1):.1f}% of HD-BET mask); "
            f"LINDA added {added:,} voxels HD-BET didn't have. "
            f"With the brain_mask= bypass, these should be ~identical."
            f"</small>"
        ))

    # Stage 1: Original + HD-BET mask
    if t1_orig.exists() and hdbet_mask.exists():
        display(HTML(
            "<hr><b>1. Original T1 with HD-BET brain mask (red)</b><br>"
            "<small style='color:#aaa'>What HD-BET says is brain. "
            "Should cover the whole brain including cerebellum + brainstem."
            "</small>"
        ))
        nv1 = NiiVue()
        nv1.load_volumes([
            {"path": str(t1_orig)},
            {"path": str(hdbet_mask), "colormap": "red", "opacity": 0.40},
        ])
        display(nv1)

    # Stage 2: LINDA's N4 brain + BrainMask
    if linda_n4_brain.exists() and linda_brainmask.exists():
        display(HTML(
            "<hr><b>2. LINDA's N4-corrected brain + BrainMask (red)</b><br>"
            "<small style='color:#aaa'>What LINDA produced. With the "
            "brain_mask= bypass, this red mask should match HD-BET's "
            "mask above voxel-for-voxel — LINDA wrote our mask through "
            "without re-stripping."
            "</small>"
        ))
        nv2 = NiiVue()
        nv2.load_volumes([
            {"path": str(linda_n4_brain)},
            {"path": str(linda_brainmask), "colormap": "red", "opacity": 0.40},
        ])
        display(nv2)

    # Stage 3: HD-BET vs LINDA, both overlaid on T1
    if t1_orig.exists() and hdbet_mask.exists() and linda_brainmask.exists():
        display(HTML(
            "<hr><b>3. HD-BET mask (green) vs LINDA mask (red), on original T1</b><br>"
            "<small style='color:#aaa'>Green-only = HD-BET said brain, "
            "LINDA dropped it. Red-only = LINDA added voxels HD-BET "
            "didn't have. Yellow-ish blend = both agree. With the bypass "
            "engaged you should see almost entirely yellow."
            "</small>"
        ))
        nv3 = NiiVue()
        nv3.load_volumes([
            {"path": str(t1_orig)},
            {"path": str(hdbet_mask),      "colormap": "green", "opacity": 0.35},
            {"path": str(linda_brainmask), "colormap": "red",   "opacity": 0.35},
        ])
        display(nv3)

    # Stage 4: Final lesion mask (native) on the original T1
    if t1_orig.exists() and lesion_native.exists():
        display(HTML(
            "<hr><b>4. Final lesion mask (red) on the original T1 — native space</b><br>"
            "<small style='color:#aaa'>LINDA's predicted lesion in patient "
            "anatomy. This is the segmentation you'll use for downstream "
            "analysis. If the lesion is missing or way too small, LINDA's "
            "brain-mask coverage was probably the upstream culprit — see "
            "panel 3."
            "</small>"
        ))
        nv4 = NiiVue()
        nv4.load_volumes([
            {"path": str(t1_orig)},
            {"path": str(lesion_native), "colormap": "red", "opacity": 0.6},
        ])
        display(nv4)

    # Stage 5: Lesion in MNI on Subject_in_MNI
    if subject_mni.exists() and lesion_mni.exists():
        display(HTML(
            "<hr><b>5. Lesion (red) in MNI space, on Subject_in_MNI</b><br>"
            "<small style='color:#aaa'>Same lesion warped to MNI152. "
            "Use this for group overlays and atlas-based interpretation."
            "</small>"
        ))
        nv5 = NiiVue()
        nv5.load_volumes([
            {"path": str(subject_mni)},
            {"path": str(lesion_mni), "colormap": "red", "opacity": 0.6},
        ])
        display(nv5)

,Stage,Voxels > 0,Path
✓,Original T1,"3,354,619",/neurodesktop-storage/aphasia-lesion-pipeline/ds004884_anat/sub-M2042/ses-1953/anat/sub-M2042_ses-1953_acq-tfl3p2_run-7_T1w.nii.gz
✓,"HD-BET brain (T1, skull-stripped)","1,453,525",/neurodesktop-storage/aphasia-lesion-pipeline/ds004884_anat/derivatives/lesion_mask/sub-M2042/ses-1953/anat/_hdbet_work/sub-M2042_ses-1953_acq-tfl3p2_run-7_T1w_brain.nii.gz
✓,HD-BET brain mask,"1,453,679",/neurodesktop-storage/aphasia-lesion-pipeline/ds004884_anat/derivatives/lesion_mask/sub-M2042/ses-1953/anat/_hdbet_work/sub-M2042_ses-1953_acq-tfl3p2_run-7_T1w_brain_mask.nii.gz
✗,LINDA N4_corrected_Brain,—,/neurodesktop-storage/aphasia-lesion-pipeline/ds004884_anat/derivatives/lesion_mask/sub-M2042/ses-1953/anat/linda/N4corrected_Brain.nii.gz
✓,LINDA BrainMask,"1,453,679",/neurodesktop-storage/aphasia-lesion-pipeline/ds004884_anat/derivatives/lesion_mask/sub-M2042/ses-1953/anat/linda/BrainMask.nii.gz
✓,LINDA lesion (native space),"238,778",/neurodesktop-storage/aphasia-lesion-pipeline/ds004884_anat/derivatives/lesion_mask/sub-M2042/ses-1953/anat/linda/Prediction3_native.nii.gz
✓,LINDA lesion (MNI space),"288,108",/neurodesktop-storage/aphasia-lesion-pipeline/ds004884_anat/derivatives/lesion_mask/sub-M2042/ses-1953/anat/linda/Lesion_in_MNI.nii.gz
✓,LINDA Subject in MNI,"1,816,453",/neurodesktop-storage/aphasia-lesion-pipeline/ds004884_anat/derivatives/lesion_mask/sub-M2042/ses-1953/anat/linda/Subject_in_MNI.nii.gz


In [ ]:
# ============================================================
# Run LINDA on every discovered T1w
# ------------------------------------------------------------
# Default pipeline (CONFIG["USE_HDBET_PREPROCESSING"] = True):
#   for each subject:
#     HD-BET → padded brain T1 → LINDA → outputs in canonical linda/
#
# Plain-LINDA pipeline (CONFIG["USE_HDBET_PREPROCESSING"] = False):
#   for each subject:
#     LINDA on raw T1 → outputs in canonical linda/
#
# Already-segmented subjects are skipped unless CONFIG["OVERWRITE"]
# is True.
# ============================================================
import sys, importlib, subprocess
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import linda_qc as q
importlib.reload(q)


if not CONFIG["RUN_SEGMENTATION"]:
    print("RUN_SEGMENTATION = False — skipping segmentation.")
else:
    use_hdbet = bool(CONFIG.get("USE_HDBET_PREPROCESSING", True))
    overwrite = bool(CONFIG.get("OVERWRITE", False))

    if use_hdbet and q.detect_brain_extractor() is None:
        print("❌ USE_HDBET_PREPROCESSING is True but no brain extractor "
              "is on PATH.")
        print()
        print(q.brain_extractor_help_text())
        raise SystemExit(
            "Run the brain-extractor module-load cell, then re-run "
            "this cell."
        )

    print(f"Running LINDA on {len(SUBJECTS)} subject(s)")
    print(f"  HD-BET preprocessing: {use_hdbet}")
    if use_hdbet:
        print(f"  HD-BET mode: {CONFIG.get('HDBET_MODE','fast')}, "
              f"padding: {CONFIG.get('HDBET_PADDING_MM',8.0)}mm")
    print(f"  Overwrite existing: {overwrite}")
    print()

    n_done, n_skipped, n_failed = 0, 0, 0
    for entry in SUBJECTS:
        out_dir = deriv_path_for(entry)
        out_dir.mkdir(parents=True, exist_ok=True)
        expected = out_dir / "Lesion_in_MNI.nii.gz"
        tag = f"{entry['subject']} {entry['session'] or ''}".strip()

        if expected.exists() and expected.stat().st_size > 0 and not overwrite:
            print(f"  ✓ {tag}: already done — skipping")
            n_skipped += 1
            continue

        print(f"  ▶ {tag}: running on {entry['t1w'].name}")
        try:
            if use_hdbet:
                res = q.hdbet_then_linda(
                    entry["t1w"], out_dir,
                    use_mask_bypass=bool(CONFIG.get("HDBET_USE_MASK_BYPASS", True)),
                    use_padding=True,
                    padding_mm=float(CONFIG.get("HDBET_PADDING_MM", 8.0)),
                    csf_rim_mm=float(CONFIG.get("HDBET_CSF_RIM_MM", 2.0)),
                    hdbet_mode=str(CONFIG.get("HDBET_MODE", "fast")),
                    hdbet_device=CONFIG.get("HDBET_DEVICE"),
                )
                rc = res.get("linda_returncode")
            else:
                rc = subprocess.run(
                    ["linda_predict.sh", str(entry["t1w"]), str(out_dir)],
                    check=False,
                ).returncode

            if expected.exists() and expected.stat().st_size > 0:
                print(f"    ✓ done (LINDA rc={rc})")
                n_done += 1
            else:
                print(f"    ❌ LINDA finished (rc={rc}) but no "
                      f"Lesion_in_MNI.nii.gz")
                n_failed += 1
        except Exception as e:
            print(f"    ❌ failed: {e}")
            n_failed += 1

    print(f"\nSegmentation summary: {n_done} new, {n_skipped} skipped, "
          f"{n_failed} failed (out of {len(SUBJECTS)})")


### Relocate LINDA outputs

In some LINDA versions, the predictor writes a `linda/` folder *next to the input
T1w*. The cell below moves any such folders into the BIDS-derivatives layout used
by the rest of the notebook. If your LINDA run already wrote into `DERIV_DIR`,
this is a no-op.


In [ ]:
# ============================================================
# Move stray linda/ folders that LINDA may have written next to the
# input T1w into the canonical derivatives location.
# ============================================================
moved = 0
for t1 in ANAT_DIR.rglob("*_T1w.nii.gz"):
    src = t1.parent / "linda"
    if not src.is_dir():
        continue
    rel = t1.parent.relative_to(ANAT_DIR)  # sub-*/(ses-*/)anat
    dst = DERIV_DIR / rel / "linda"
    if dst.exists():
        continue
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(src), str(dst))
    print(f"  ↪ moved {src} → {dst}")
    moved += 1

if moved == 0:
    print("No stray linda/ folders to relocate — derivatives already in place.")
else:
    print(f"Relocated {moved} folder(s)")


In [ ]:
# Quick check: how many subjects have a Lesion_in_MNI.nii.gz?
have_lesion = []
missing = []
for entry in SUBJECTS:
    if lesion_mni_for(entry).exists():
        have_lesion.append(entry)
    else:
        missing.append(entry)

print(f"Lesion masks present: {len(have_lesion)}/{len(SUBJECTS)}")
if missing and CONFIG["VERBOSE"]:
    print("Missing for:")
    for e in missing[:10]:
        print(f"  - {e['subject']} {e['session'] or ''}")
    if len(missing) > 10:
        print(f"  … and {len(missing) - 10} more")


## Results & visualization

The cells below build on the `SUBJECTS` list and `CONFIG`. Each part is
independent — you can re-run the QC dashboard, the group overlap map, or the
atlas overlap analysis on its own.


### Unified QC dashboard

A single interactive viewer for inspecting LINDA outputs. The dropdown lets you
jump between subjects; the tabs switch between skull-stripping QC, lesion in
native space, lesion in MNI space, and the atlas overlay. Subjects without LINDA
output are omitted so the dashboard never crashes on missing files.


In [ ]:
# ============================================================
# Stage-aware QC dashboard + rating widget
# ------------------------------------------------------------
# Pick a stage and a subject. The viewer + rating UI rebuild fresh on
# every change. Each subject starts UNRATED ("—"); Save refuses to
# write until you pick 1 / 2 / 3.
#
# Workflow: judge skull strip first. If poor, run the HD-BET cell BELOW
# this one (with the subject's index, or HDBET_BATCH=True for all
# marked-for-rerun subjects) before judging the lesion.
#
# Full rubric: ../LINDA-STUFF/QC_RUBRIC.md
# ============================================================
import sys
import importlib
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import linda_qc as q
importlib.reload(q)

import datetime as _dt
import ipywidgets as widgets
from ipyniivue import NiiVue
from IPython.display import display, HTML


# ---------------- VISUALIZATION CONFIG ----------------
# (Edit these values to taste; no kernel restart needed for the QC cell —
#  just re-run the cell after changing.)
LESION_OPACITY            = 0.80   # 0=invisible, 1=opaque
LESION_AS_BOUNDARY        = True   # True = show only the lesion's edge ring
LESION_BOUNDARY_THICKNESS = 1      # voxels wide (1=thin, 2=medium, 3=thick)
                                   # Each thickness is cached separately as
                                   # <mask>_boundary_tN.nii.gz next to the mask.
BRAIN_MASK_OPACITY        = 0.30   # opacity for the skull-strip overlay
# ------------------------------------------------------


qc_pool = [e for e in SUBJECTS if lesion_mni_for(e).exists()]
qc_labels = [f"[{i}] {e['subject']} {e['session'] or ''}".strip()
             for i, e in enumerate(qc_pool)]


def _lesion_volume_path(d):
    """Resolve to either the raw mask or the cached boundary."""
    raw = d / "Lesion_in_MNI.nii.gz"
    if not raw.exists():
        return None
    if LESION_AS_BOUNDARY:
        return q.ensure_lesion_boundary(raw, thickness=LESION_BOUNDARY_THICKNESS)
    return raw


def _build_vols(stage, e):
    d = deriv_path_for(e)
    t1 = e["t1w"]
    vols = []
    if stage == "skull_strip":
        if (d / "N4corrected.nii.gz").exists():
            vols.append({"path": str(d / "N4corrected.nii.gz")})
        elif Path(t1).exists():
            vols.append({"path": str(t1)})
        if (d / "BrainMask.nii.gz").exists():
            vols.append({"path": str(d / "BrainMask.nii.gz"),
                         "colormap": "red", "opacity": BRAIN_MASK_OPACITY})
    else:  # lesion
        if (d / "Subject_in_MNI.nii.gz").exists():
            vols.append({"path": str(d / "Subject_in_MNI.nii.gz")})
        lp = _lesion_volume_path(d)
        if lp is not None:
            vols.append({"path": str(lp),
                         "colormap": "red", "opacity": LESION_OPACITY})
    return vols


def _stage_chips(rec):
    chips = []
    for st in q.STAGES:
        r = (rec.stages.get(st) or {}).get("rating")
        color = ({1: "#3aaf3a", 2: "#e6a23c", 3: "#d9534f"}.get(r, "#888"))
        label = "—" if r is None else f"{r}"
        chips.append(
            f"<span style='display:inline-block;background:{color};"
            f"color:white;padding:2px 8px;border-radius:10px;"
            f"margin-right:6px;font-size:11px'>"
            f"{q.STAGE_LABELS[st].split(' ')[0]}: {label}</span>"
        )
    return "".join(chips)


def qc_panel(stage, idx):
    """Driven by widgets.interact below. Rebuilds the entire view."""
    if not qc_pool:
        display(HTML("<em>No lesion masks yet — run segmentation first.</em>"))
        return
    if not 0 <= idx < len(qc_pool):
        display(HTML(f"<em>Index {idx} out of range (0..{len(qc_pool)-1}).</em>"))
        return

    e = qc_pool[idx]
    rec = q.QCRecord.load(lesion_mni_for(e))
    sd = rec.stages.get(stage) or {}

    display(HTML(
        f"<h4 style='margin-bottom:4px'>"
        f"[{idx}] {e['subject']} {e['session'] or ''} "
        f"&nbsp;·&nbsp; <span style='color:#888'>{q.STAGE_LABELS[stage]}</span>"
        f"</h4>"
        f"<div style='margin-bottom:8px'>{_stage_chips(rec)}</div>"
    ))

    # Viewer
    vols = _build_vols(stage, e)
    if not vols:
        display(HTML("<em>No volumes available for this stage.</em>"))
    else:
        nv = NiiVue()
        nv.load_volumes(vols)
        display(nv)

    # File-path debug, with modification times so you can see at a
    # glance whether the files are fresh after a re-run.
    import datetime as _dt_dbg
    def _mtime_str(p):
        try:
            ts = _dt_dbg.datetime.fromtimestamp(Path(p).stat().st_mtime)
            return ts.strftime("%Y-%m-%d %H:%M")
        except Exception:
            return "?"
    display(HTML(
        "<small style='color:#888; font-family:monospace'>"
        + "<br>".join(
            f"  · {v['path']}"
            f" <span style='color:#aaa'>(modified {_mtime_str(v['path'])})</span>"
            for v in vols
        )
        + "</small>"
    ))

    # Rating + tags + notes + save
    stored = sd.get("rating")
    rating = widgets.ToggleButtons(
        options=[("—", None), ("1 · good", 1),
                 ("2 · acceptable", 2), ("3 · poor", 3)],
        value=stored if stored in (1, 2, 3) else None,
        description="Rating:",
        style={"description_width": "initial"},
    )
    _, ratings_for_stage = q.STAGE_VOCAB[stage]
    rating_help = widgets.HTML()

    def _refresh_rating_help(*_):
        if rating.value is None:
            rating_help.value = (
                "<small style='color:#d9534f'>"
                "<b>Not rated yet.</b> Pick 1, 2, or 3, then click Save."
                "</small>"
            )
        else:
            defn = ratings_for_stage.get(rating.value, "")
            rating_help.value = (
                f"<small style='color:#aaa'>"
                f"<b>{rating.value} · {q.RATING_VOCAB[rating.value]}:</b> "
                f"{defn}</small>"
            )
    rating.observe(_refresh_rating_help, names="value")
    _refresh_rating_help()

    tag_dict, _ = q.STAGE_VOCAB[stage]
    issues = widgets.SelectMultiple(
        options=list(tag_dict.keys()),
        value=tuple(t for t in (sd.get("issue_tags") or []) if t in tag_dict),
        rows=min(8, len(tag_dict)),
        description="Issues:",
        layout=widgets.Layout(width="380px"),
        style={"description_width": "initial"},
    )

    notes = widgets.Textarea(
        value=sd.get("notes") or "",
        description="Notes:",
        placeholder="Free-text notes for this stage...",
        layout=widgets.Layout(width="600px", height="60px"),
        style={"description_width": "initial"},
    )
    reviewer = widgets.Text(
        value=globals().get("LAST_REVIEWER", rec.reviewer or ""),
        description="Reviewer:",
        layout=widgets.Layout(width="280px"),
        style={"description_width": "initial"},
    )
    rerun_cb = widgets.Checkbox(value=bool(rec.marked_for_rerun),
                                 description="Mark for re-run")
    save_btn = widgets.Button(description="Save QC", button_style="primary")
    save_msg = widgets.Output()

    def _save(_):
        if rating.value is None:
            with save_msg:
                save_msg.clear_output()
                display(HTML(
                    "<b style='color:#d9534f'>⚠ Pick a rating (1/2/3) "
                    "before saving.</b>"
                ))
            return
        rec.subject  = e["subject"]
        rec.session  = e["session"]
        rec.set_stage(stage,
                      rating=rating.value,
                      issue_tags=list(issues.value),
                      notes=notes.value.strip())
        rec.reviewer = reviewer.value.strip() or rec.reviewer
        rec.reviewed_on = _dt.date.today().isoformat()
        rec.marked_for_rerun = bool(rerun_cb.value)
        path = rec.save()
        globals()["LAST_REVIEWER"] = rec.reviewer
        with save_msg:
            save_msg.clear_output()
            display(HTML(
                f"<b style='color:#3aaf3a'>✓ Saved {q.STAGE_LABELS[stage]}</b> "
                f"for <code>{e['subject']} {e['session'] or ''}</code> "
                f"→ <code>{path.name}</code><br>"
                f"<small>Move the slider to advance.</small>"
            ))
    save_btn.on_click(_save)

    display(widgets.HTML("<hr><b>Rate this stage</b>"))
    display(rating, rating_help, issues, notes,
            widgets.HBox([reviewer, rerun_cb, save_btn]),
            save_msg)



def _existing_qc_banner(existing_pairs, slider_widget, stage_widget):
    """Show an interactive 'found existing ratings' banner. Returns
    the Output widget that hosts it.

    `slider_widget` and `stage_widget` are the explicit ipywidgets we
    pass to interact; we use them to nudge a re-render after clearing.
    """
    n = len(existing_pairs)
    out = widgets.Output()

    def _wrap(content_html, color_bg, color_border, color_text="#333"):
        # Explicit text color so the banner reads on dark themes.
        return (
            f"<div style='background:{color_bg}; border:1px solid {color_border}; "
            f"color:{color_text}; "
            f"border-radius:6px; padding:10px 14px; margin-bottom:10px'>"
            f"{content_html}</div>"
        )

    def _nudge():
        """Force qc_panel to re-render by toggling slider value."""
        if slider_widget.max > 0:
            cur = slider_widget.value
            other = (cur + 1) if cur < slider_widget.max else (cur - 1)
            slider_widget.value = other
            slider_widget.value = cur
        elif len(stage_widget.options) > 1:
            cur_stage = stage_widget.value
            other = next(o[1] for o in stage_widget.options
                         if o[1] != cur_stage)
            stage_widget.value = other
            stage_widget.value = cur_stage

    def _initial():
        with out:
            out.clear_output()
            display(HTML(_wrap(
                f"<b>ℹ Found {n} existing QC rating(s)</b> "
                f"for subjects in this dataset (loaded from "
                f"<code style='background:#eee; padding:1px 4px; "
                f"border-radius:3px; color:#222'>*.qc.json</code> sidecars).",
                "#fff3cd", "#ffe69c",
            )))
            keep_btn  = widgets.Button(description="Keep & continue",
                                       button_style="primary")
            clear_btn = widgets.Button(description="Clear all (keep edit history)",
                                       button_style="warning")
            show_btn  = widgets.Button(description="Show details")
            keep_btn.on_click(lambda _: _dismiss())
            clear_btn.on_click(lambda _: _confirm_clear())
            show_btn.on_click(lambda _: _show_details())
            display(widgets.HBox([keep_btn, clear_btn, show_btn]))

    def _show_details():
        with out:
            out.clear_output()
            rows = []
            for e, sp in existing_pairs:
                rec = q.QCRecord.load(lesion_mni_for(e))
                stages = []
                for st in q.STAGES:
                    r = (rec.stages.get(st) or {}).get("rating")
                    label = q.STAGE_LABELS[st].split(" ")[0]
                    stages.append(f"{label}: <b>{r if r is not None else '—'}</b>")
                rows.append(
                    f"<li><code style='background:#eee; padding:1px 4px; "
                    f"border-radius:3px; color:#222'>"
                    f"{e['subject']} {e['session'] or ''}</code> "
                    f"&nbsp; {' &nbsp; '.join(stages)} "
                    f"&nbsp; <small style='color:#666'>"
                    f"reviewer: {rec.reviewer or '—'}</small></li>"
                )
            display(HTML(_wrap(
                f"<b>Existing QC ratings ({n})</b>"
                f"<ul style='margin:6px 0 0 18px; padding:0; "
                f"font-size:12px'>{''.join(rows)}</ul>",
                "#fff3cd", "#ffe69c",
            )))
            back = widgets.Button(description="Back")
            back.on_click(lambda _: _initial())
            display(back)

    def _confirm_clear():
        with out:
            out.clear_output()
            display(HTML(_wrap(
                f"<b>⚠ Clear all {n} QC rating(s)?</b><br>"
                f"<small>Per-edit history (deterministic ops, paint, HD-BET) "
                f"is preserved — only ratings, tags, notes, and reviewer fields "
                f"are cleared. For full sidecar deletion, use the "
                f"<code style='background:#eee; padding:1px 4px; "
                f"border-radius:3px; color:#222'>Reset QC</code> cell below "
                f"with <code style='background:#eee; padding:1px 4px; "
                f"border-radius:3px; color:#222'>KEEP_EDITS=False</code>."
                f"</small>",
                "#f8d7da", "#f5c2c7", color_text="#842029",
            )))
            yes = widgets.Button(description="Yes, clear ratings",
                                 button_style="danger")
            no  = widgets.Button(description="Cancel", button_style="primary")
            yes.on_click(lambda _: _do_clear())
            no.on_click(lambda _: _initial())
            display(widgets.HBox([yes, no]))

    def _do_clear():
        affected = q.reset_qc_sidecars(
            DERIV_DIR, keep_edits=True, dry_run=False,
        )
        with out:
            out.clear_output()
            display(HTML(_wrap(
                f"<b>✓ Cleared {len(affected)} QC rating(s).</b> "
                f"The panel below has been refreshed; subjects now show as "
                f"unrated.",
                "#d1e7dd", "#a3cfbb", color_text="#0f5132",
            )))
        # Force the panel below to re-render with the cleared state.
        _nudge()

    def _dismiss():
        with out:
            out.clear_output()
            display(HTML(_wrap(
                f"<small>Existing ratings preserved. "
                f"({n} sidecars carried over.)</small>",
                "#e9ecef", "#dee2e6", color_text="#495057",
            )))

    _initial()
    return out


if not qc_pool:
    display(HTML("<em>No lesion masks yet.</em>"))
else:
    # Detect existing QC sidecars and offer a one-click clear before
    # showing the interactive widget.
    _existing_pairs = []
    for _e in qc_pool:
        _sp = q.sidecar_path_for(lesion_mni_for(_e))
        if _sp.exists():
            _existing_pairs.append((_e, _sp))

    # Explicit slider + toggle so we can nudge them after clearing.
    _stage_toggle = widgets.ToggleButtons(
        options=[(q.STAGE_LABELS[s], s) for s in q.STAGES],
        description="Stage:",
        style={"description_width": "initial"},
    )
    _subject_slider = widgets.IntSlider(
        min=0, max=len(qc_pool) - 1, value=0,
        description=f"Subject (0–{len(qc_pool) - 1}):",
        continuous_update=False,
        style={"description_width": "initial"},
        layout=widgets.Layout(width="500px"),
    )

    if _existing_pairs:
        display(_existing_qc_banner(
            _existing_pairs, _subject_slider, _stage_toggle))

    widgets.interact(qc_panel, stage=_stage_toggle, idx=_subject_slider)


In [ ]:
# ============================================================
# HD-BET re-strip + LINDA re-run (skull-strip repair)
# ------------------------------------------------------------
# Three modes via HDBET_MODE:
#
#   "single"       — process only HDBET_SUBJECT_IDX (the [N] from the
#                    QC slider header above).
#   "step_marked"  — process the next subject with marked_for_rerun=True.
#                    Re-run the cell to advance to the next one.
#                    On a successful run (DRY_RUN=False) the flag is
#                    cleared, so the queue naturally shrinks.
#   "batch_marked" — process every marked-for-rerun subject in one go.
#
# Pre-requisite: HD-BET (or SynthStrip) must be on PATH. Run the
# brain-extractor module-load cell near the top of the notebook first.
# ============================================================
import sys
import importlib
import json
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import linda_qc as q
importlib.reload(q)

import ipywidgets as widgets
from ipyniivue import NiiVue
from IPython.display import display, HTML

# ----------------------------------------------------------------
# CONFIG
HDBET_MODE        = "step_marked"   # "single" | "step_marked" | "batch_marked"
HDBET_SUBJECT_IDX = 0          # only used when HDBET_MODE == "single"
DRY_RUN           = False       # True  = brain extraction only, no LINDA re-run
                               # False = full re-strip + LINDA re-run
                               #         (clears marked_for_rerun on success)

# HD-BET runtime knobs (HD-BET only — SynthStrip ignores these)
HDBET_DEVICE = None            # None = auto (GPU if available, else CPU)
HDBET_MODE_HDBET = "fast"      # "fast" (~30s-2min CPU, may drop
                               #         cerebellum/brainstem) or
                               #         "accurate" (5-model ensemble,
                               #         much better coverage but ~10x
                               #         slower on CPU). If you see a
                               #         MASK COVERAGE WARNING after a
                               #         run, switch to "accurate".
HDBET_TTA    = False           # test-time augmentation

# Padding (fake-skull rim around the HD-BET brain). LINDA's own brain
# stripper tends to over-strip a bare HD-BET image (observed: brain
# mask shrinks to ~47% of original). With padding=True we wrap the
# brain in a low-intensity rim that LINDA strips off, leaving the
# brain intact.
HDBET_USE_PADDING = True       # True = wrap brain in fake-skull rim
                               # before handing to LINDA (recommended)
HDBET_PADDING_MM  = 8.0        # rim thickness in mm
# ----------------------------------------------------------------

qc_pool = [e for e in SUBJECTS if lesion_mni_for(e).exists()]
extractor = q.detect_brain_extractor()


def _marked_for_rerun_subjects():
    out = []
    for e in qc_pool:
        sp = q.sidecar_path_for(lesion_mni_for(e))
        if not sp.exists():
            continue
        try:
            data = json.loads(sp.read_text())
        except Exception:
            continue
        if data.get("marked_for_rerun"):
            out.append(e)
    return out


def _show_brain_mask_viewer(t1_path, brain_mask_path):
    """Inline viewer of T1 with the new brain mask overlay."""
    nv = NiiVue()
    nv.load_volumes([
        {"path": str(t1_path)},
        {"path": str(brain_mask_path),
         "colormap": "red", "opacity": 0.30},
    ])
    display(HTML("<small style='color:#aaa'>"
                 "T1 with new brain mask overlay (HD-BET / SynthStrip)"
                 "</small>"))
    display(nv)


def _run_one(e):
    """Run HD-BET (and optionally LINDA) on one subject; show viewer."""
    print(f"→ {e['subject']} {e['session'] or ''}")
    print(f"  T1w:    {e['t1w']}")
    print(f"  Output: {deriv_path_for(e)}")

    if DRY_RUN:
        res = q.run_hd_bet(
            e["t1w"],
            PROJECT_DIR / "_hdbet_work" /
            f"{e['subject']}_{e['session'] or 'no-ses'}",
            device=HDBET_DEVICE, mode=HDBET_MODE_HDBET, tta=HDBET_TTA,
        )
        print(f"  brain extraction rc={res['returncode']}")
        if res["returncode"] == 0 and res["mask"].exists():
            _show_brain_mask_viewer(e["t1w"], res["mask"])
    else:
        res = q.restrip_and_rerun(
            e["t1w"], deriv_path_for(e),
            reviewer=globals().get("LAST_REVIEWER"),
            use_mask_bypass=bool(CONFIG.get("HDBET_USE_MASK_BYPASS", True)),
            use_padding=HDBET_USE_PADDING,
            padding_mm=HDBET_PADDING_MM,
            csf_rim_mm=float(CONFIG.get("HDBET_CSF_RIM_MM", 2.0)),
            rscript_cmd=str(CONFIG.get("HDBET_RSCRIPT_CMD", "Rscript")),
        )
        print(f"  brain extractor rc={res.get('returncode')}, "
              f"LINDA rc={res.get('linda_returncode')}")
        regen = res.get("regenerated_files") or []
        missing = res.get("missing_after_rerun") or []
        warning = res.get("mask_warning")
        if regen:
            print(f"  ✓ Regenerated {len(regen)} canonical file(s): "
                  f"{', '.join(regen)}")
        else:
            print(f"  ⚠ NO canonical files were regenerated. LINDA may "
                  f"have failed silently or its outputs were not cleared. "
                  f"Check the LINDA output above.")
        if missing:
            print(f"  ⚠ Missing after re-run: {', '.join(missing)}")
        if warning:
            print()
            print(f"  ⚠ MASK COVERAGE WARNING:")
            print(f"     {warning}")
        if regen:
            print()
            print(f"  ↻ QC ratings were RESET for this subject — the new")
            print(f"     images need fresh review. Old ratings preserved")
            print(f"     in the sidecar's edits[] list (audit trail).")
        if res.get("returncode") == 0 and res.get("mask") and res["mask"].exists():
            _show_brain_mask_viewer(e["t1w"], res["mask"])
    print()


# ============================================================
# Dispatch
# ============================================================
if extractor is None:
    print(q.brain_extractor_help_text())
elif HDBET_MODE not in ("single", "step_marked", "batch_marked"):
    print(f"Unknown HDBET_MODE={HDBET_MODE!r}; "
          f"use 'single', 'step_marked', or 'batch_marked'.")
else:
    has_gpu = q._has_nvidia_gpu()
    eff_device = HDBET_DEVICE if HDBET_DEVICE is not None else (
        "0" if has_gpu else "cpu"
    )

    print(f"Mode:       {HDBET_MODE}")
    print(f"Extractor:  {extractor}")
    print(f"Padding:    {'ON ('+str(HDBET_PADDING_MM)+'mm rim)' if HDBET_USE_PADDING else 'OFF (bare brain)'}")
    if extractor == "hd-bet":
        print(f"Device:     {eff_device}  ({'GPU' if has_gpu else 'CPU'})")
    print(f"DRY_RUN:    {DRY_RUN}  "
          f"({'extraction only' if DRY_RUN else 'extraction + LINDA re-run'})")
    print()

    if HDBET_MODE == "single":
        if not 0 <= HDBET_SUBJECT_IDX < len(qc_pool):
            print(f"HDBET_SUBJECT_IDX={HDBET_SUBJECT_IDX} out of range "
                  f"(0..{len(qc_pool)-1}).")
        else:
            _run_one(qc_pool[HDBET_SUBJECT_IDX])

    else:
        # Both step and batch start by listing the queue
        marked = _marked_for_rerun_subjects()
        if not marked:
            print("No subjects with marked_for_rerun=True.")
            print("In the QC widget above, check 'Mark for re-run' for any "
                  "subject you want to include, then re-run this cell.")
        else:
            print(f"Queue ({len(marked)} subject(s) marked for re-run):")
            for e in marked:
                print(f"  · {e['subject']} {e['session'] or ''}")
            print()

            if HDBET_MODE == "step_marked":
                e = marked[0]
                print(f"STEP MODE — processing the first one. "
                      f"({len(marked) - 1} more remain after this run.)")
                print()
                _run_one(e)
                if DRY_RUN:
                    print("DRY_RUN=True — flag NOT cleared. Set DRY_RUN=False")
                    print("to apply and clear the flag, then re-run this cell")
                    print("to advance to the next marked subject.")
                else:
                    remaining = len(_marked_for_rerun_subjects())
                    if remaining:
                        print(f"✓ Done with this subject. "
                              f"{remaining} more marked subject(s) remain. "
                              f"Re-run this cell to process the next.")
                    else:
                        print("✓ Queue empty — all marked subjects processed.")

            elif HDBET_MODE == "batch_marked":
                print(f"BATCH MODE — processing all {len(marked)} subjects.")
                print()
                for e in marked:
                    _run_one(e)
                print(f"Done. Processed {len(marked)} subject(s).")
                if DRY_RUN:
                    print("DRY_RUN=True — set DRY_RUN=False and re-run to apply.")


### Manual fixes & re-run

Heavier tools for repairing or re-running specific masks. Use these
when the in-line rating widget above flagged a mask as poor and the
issue can't be resolved with simple post-processing.

1. **Tier 1 — deterministic post-processing.** Threshold, morphology,
   hemisphere clip, largest connected component. Reproducible and
   logged.
2. **Tier 2 — NiiVue draw mode.** In-notebook voxel painting.
3. **Tier 3 — external editor.** Round-trip to ITK-SNAP / FSLeyes /
   MRIcroGL.
4. **Re-run queue.** Subjects you marked for re-run, optionally with
   the (corrected) current mask as a manual seed.
5. **QC summary.** Aggregate every sidecar into `qc_summary.csv`
   plus rating/issue/queue charts.

Every original LINDA output is preserved in `<linda_dir>/_linda_original/`
on the first edit; the canonical `Lesion_in_MNI.nii.gz` is updated in
place so the rest of the notebook (atlas overlap, group map, reports)
sees the latest edited version.


In [ ]:
# === QC WORKFLOW BLOCK START ===
# (rating UI is in the unified QC cell above.)
# ============================================================
# Step 2 — Tier 1: deterministic post-processing
# Live preview + Apply & save (with backup of LINDA original)
# ============================================================
import sys, importlib
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import linda_qc as q
importlib.reload(q)

import ipywidgets as widgets
import nibabel as nib
import numpy as np
from IPython.display import display, HTML, clear_output

try:
    from ipyniivue import NiiVue
    HAVE_NV = True
except Exception:
    HAVE_NV = False

if not qc_pool:
    display(HTML("<em>No masks to fix.</em>"))
else:
    subj_dd2 = widgets.Dropdown(
        options=[f"{e['subject']}{' ' + e['session'] if e['session'] else ''}"
                 for e in qc_pool],
        description="Subject:",
        layout=widgets.Layout(width="320px"),
        style={"description_width": "initial"},
    )
    thresh_sl = widgets.FloatSlider(value=0.5, min=0.1, max=0.95, step=0.05,
                                    description="Threshold:", continuous_update=False,
                                    style={"description_width": "initial"})
    open_sl   = widgets.IntSlider(value=0, min=0, max=3, step=1,
                                  description="Morph open:", continuous_update=False,
                                  style={"description_width": "initial"})
    close_sl  = widgets.IntSlider(value=0, min=0, max=3, step=1,
                                  description="Morph close:", continuous_update=False,
                                  style={"description_width": "initial"})
    fill_cb   = widgets.Checkbox(value=False, description="Fill holes")
    cc_sl     = widgets.IntSlider(value=0, min=0, max=5, step=1,
                                  description="Keep N largest CC:", continuous_update=False,
                                  style={"description_width": "initial"})
    hemi_dd   = widgets.Dropdown(options=["none", "left", "right"],
                                 value="none", description="Hemisphere clip:",
                                 style={"description_width": "initial"})
    reviewer2 = widgets.Text(value=globals().get("LAST_REVIEWER", ""),
                             description="Reviewer:",
                             style={"description_width": "initial"})
    apply_btn  = widgets.Button(description="Apply & save",
                                button_style="primary")
    revert_btn = widgets.Button(description="Revert to LINDA original",
                                button_style="warning")
    out = widgets.Output()
    preview_out = widgets.Output()

    def _entry():
        return next(e for e in qc_pool
                    if (f"{e['subject']}{' ' + e['session'] if e['session'] else ''}"
                        == subj_dd2.value))

    def _ops_list():
        ops = [{"name": "threshold", "params": {"threshold": thresh_sl.value}}]
        if open_sl.value:  ops.append({"name": "morph_open",  "params": {"radius": open_sl.value}})
        if close_sl.value: ops.append({"name": "morph_close", "params": {"radius": close_sl.value}})
        if fill_cb.value:  ops.append({"name": "fill_holes",  "params": {}})
        if cc_sl.value:    ops.append({"name": "largest_cc",  "params": {"n_components": cc_sl.value}})
        if hemi_dd.value != "none":
            ops.append({"name": "hemisphere_clip",
                        "params": {"hemisphere": hemi_dd.value}})
        return ops

    def _preview(*_):
        e = _entry()
        # Always preview from the LINDA original so adjustments are
        # cumulative against the source, not against the last save.
        src = q.linda_original_dir(lesion_mni_for(e)) / lesion_mni_for(e).name
        if not src.exists():
            src = lesion_mni_for(e)
        img = nib.load(str(src)); data = img.get_fdata(); aff = img.affine
        for op in _ops_list():
            fn = q.OP_REGISTRY[op["name"]]
            data = fn(data, affine=aff, **(op["params"] or {}))
        n_vox = int((data > 0).sum())
        with out:
            clear_output()
            display(HTML(
                f"<small>Preview: <b>{n_vox}</b> voxel(s) above threshold "
                f"with {len(_ops_list())} op(s) applied. "
                f"Click <b>Apply &amp; save</b> to commit.</small>"
            ))

    def _apply(_):
        e = _entry()
        path = lesion_mni_for(e)
        # Apply ops starting from the LINDA original (not the current file)
        src = q.linda_original_dir(path) / path.name
        if not src.exists():
            src = path
        q.apply_ops_and_save(path, _ops_list(),
                             reviewer=reviewer2.value.strip(),
                             tool="deterministic_ops",
                             source_path=src)
        with out:
            clear_output()
            display(HTML(f"<b>✓ Saved</b> {path.name} (original preserved in "
                         f"<code>_linda_original/</code>)"))

    def _revert(_):
        e = _entry()
        path = lesion_mni_for(e)
        backup = q.linda_original_dir(path) / path.name
        if not backup.exists():
            with out:
                clear_output()
                display(HTML("<em>No backup found — this mask was never edited.</em>"))
            return
        import shutil
        shutil.copy2(backup, path)
        rec = q.QCRecord.load(path)
        rec.log_edit(operation="revert_to_linda_original",
                     reviewer=reviewer2.value.strip(),
                     tool="deterministic_ops")
        rec.save()
        with out:
            clear_output()
            display(HTML(f"<b>↺ Reverted</b> {path.name}"))

    for w in (thresh_sl, open_sl, close_sl, fill_cb, cc_sl, hemi_dd, subj_dd2):
        w.observe(_preview, names="value")
    apply_btn.on_click(_apply)
    revert_btn.on_click(_revert)
    _preview()

    display(widgets.VBox([
        widgets.HBox([subj_dd2, reviewer2]),
        widgets.HBox([thresh_sl, open_sl, close_sl]),
        widgets.HBox([fill_cb, cc_sl, hemi_dd]),
        widgets.HBox([apply_btn, revert_btn]),
        out,
    ]))


In [ ]:
# ============================================================
# Step 4 — Tier 3: external-editor round-trip
# Copies T1w + mask to qc_edits/<subject>_<ses>/. After you edit and
# save in ITK-SNAP / FSLeyes, click "Import edited mask" to bring it
# back; we validate dims/affine and back up the original.
# ============================================================
import sys, importlib, shutil
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import linda_qc as q
importlib.reload(q)

import ipywidgets as widgets
from IPython.display import display, HTML

QC_EDITS_DIR = (PROJECT_DIR / "qc_edits")
QC_EDITS_DIR.mkdir(exist_ok=True)

if not qc_pool:
    display(HTML("<em>No masks available.</em>"))
else:
    subj_dd4 = widgets.Dropdown(
        options=[f"{e['subject']}{' ' + e['session'] if e['session'] else ''}"
                 for e in qc_pool],
        description="Subject:",
        layout=widgets.Layout(width="320px"),
        style={"description_width": "initial"},
    )
    reviewer4 = widgets.Text(value=globals().get("LAST_REVIEWER", ""),
                             description="Reviewer:",
                             style={"description_width": "initial"})
    editor_dd = widgets.Dropdown(options=["ITK-SNAP", "FSLeyes", "MRIcroGL"],
                                 value="ITK-SNAP", description="Editor:")
    export_btn = widgets.Button(description="Export for editing",
                                button_style="info")
    import_btn = widgets.Button(description="Import edited mask",
                                button_style="primary")
    out = widgets.Output()

    def _entry():
        return next(e for e in qc_pool
                    if (f"{e['subject']}{' ' + e['session'] if e['session'] else ''}"
                        == subj_dd4.value))

    def _export(_):
        e = _entry()
        d = deriv_path_for(e)
        edit_dir = QC_EDITS_DIR / f"{e['subject']}_{e['session'] or 'no-ses'}"
        edit_dir.mkdir(parents=True, exist_ok=True)
        copies = []
        for src in (d / "Subject_in_MNI.nii.gz", lesion_mni_for(e)):
            if src.exists():
                dst = edit_dir / src.name
                shutil.copy2(src, dst)
                copies.append(dst)
        with out:
            out.clear_output()
            print(f"✓ Exported to {edit_dir}")
            print()
            cmds = {
                "ITK-SNAP": f"itksnap -g {copies[0]} -s {copies[1]}",
                "FSLeyes":  f"fsleyes {copies[0]} {copies[1]} -cm red -a 60",
                "MRIcroGL": f"MRIcroGL {copies[0]} {copies[1]}",
            }
            print(f"Open with: {cmds[editor_dd.value]}")
            print()
            print("When done, save the lesion mask (overwrite "
                  f"{copies[1].name} in that folder) and click "
                  "'Import edited mask'.")

    def _import(_):
        e = _entry()
        edit_dir = QC_EDITS_DIR / f"{e['subject']}_{e['session'] or 'no-ses'}"
        edited = edit_dir / lesion_mni_for(e).name
        if not edited.exists():
            with out:
                out.clear_output()
                print(f"❌ {edited} not found. Did you save the edit there?")
            return
        try:
            q.import_external_edit(edited, lesion_mni_for(e),
                                   reviewer=reviewer4.value.strip(),
                                   editor=editor_dd.value)
        except ValueError as ex:
            with out: out.clear_output(); print(f"❌ {ex}"); return
        with out:
            out.clear_output()
            print(f"✓ Imported {edited.name} → {lesion_mni_for(e)}")
            print("  Original preserved in _linda_original/.")

    export_btn.on_click(_export)
    import_btn.on_click(_import)
    display(widgets.VBox([
        widgets.HBox([subj_dd4, editor_dd]),
        widgets.HBox([reviewer4, export_btn, import_btn]),
        out,
    ]))


In [ ]:
# ============================================================
# Step 5 — Re-run queue
# Lists subjects you marked for re-run. Two modes:
#   • Re-run as-is — let LINDA start from scratch on the same T1w.
#   • Re-run with manual seed — pass the *current* (likely edited)
#     Lesion_in_MNI.nii.gz as a manual seed (LINDA's 3-arg form).
# ============================================================
import sys, importlib
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import linda_qc as q
importlib.reload(q)

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

queue = q.list_rerun_queue(DERIV_DIR)
if queue.empty:
    display(HTML("<em>No subjects currently marked for re-run.</em>"))
else:
    display(HTML(f"<h4>{len(queue)} subject(s) in re-run queue</h4>"))

    out = widgets.Output()
    rows = []
    for r in queue.itertuples():
        # Find the matching SUBJECTS entry to recover the T1w path
        e = next((s for s in SUBJECTS
                  if s["subject"] == r.subject
                  and (s["session"] or "") == (r.session or "")), None)
        if e is None:
            continue
        label = widgets.Label(
            f"{r.subject}  {r.session or ''}  ·  rating={r.rating}  "
            f"·  {r.last_edit_op or 'no edits'}"
        )
        asis_btn = widgets.Button(description="Re-run as-is")
        seed_btn = widgets.Button(description="Re-run with manual seed",
                                  button_style="primary")

        def _make_handlers(entry, with_seed):
            def _h(_):
                with out:
                    clear_output()
                    seed = (lesion_mni_for(entry) if with_seed else None)
                    print(f"Re-running {entry['subject']} {entry['session'] or ''} "
                          f"({'with seed' if with_seed else 'as-is'})...")
                    rc = q.rerun_subject(
                        entry["t1w"], deriv_path_for(entry),
                        manual_seed=seed,
                        reviewer=globals().get("LAST_REVIEWER"),
                    )
                    print(f"  exit code: {rc}")
            return _h

        asis_btn.on_click(_make_handlers(e, False))
        seed_btn.on_click(_make_handlers(e, True))
        rows.append(widgets.HBox([label, asis_btn, seed_btn]))

    display(widgets.VBox(rows))
    display(out)


In [ ]:
# ============================================================
# Step 6 — QC summary
# Aggregates every sidecar into qc_summary.csv and shows a small
# dashboard of rating distribution, issue tag frequency, and edit count.
# ============================================================
import sys, importlib
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import linda_qc as q
importlib.reload(q)

import matplotlib.pyplot as plt

df = q.summarize_qc(DERIV_DIR)
csv_path = q.write_qc_summary_csv(DERIV_DIR)
print(f"QC summary written to {csv_path}")
print(f"  {len(df)} sidecar(s) found")

if not df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    # Rating distribution
    rt = df["rating"].dropna().astype(int).value_counts().reindex(
        [1, 2, 3], fill_value=0)
    axes[0].bar([f"{i} · {q.RATING_VOCAB[i]}" for i in rt.index],
                rt.values, color=["#3aaf3a", "#e6a23c", "#d9534f"])
    axes[0].set_title("Rating distribution")
    axes[0].set_ylabel("subjects")

    # Issue tag frequency
    from collections import Counter
    counter = Counter()
    for tags in df["issue_tags"].dropna():
        for t in tags.split(";"):
            t = t.strip()
            if t:
                counter[t] += 1
    if counter:
        items = counter.most_common()
        axes[1].barh([k for k, _ in items[::-1]],
                     [v for _, v in items[::-1]], color="#5078a0")
        axes[1].set_title("Issue tag frequency")
    else:
        axes[1].text(0.5, 0.5, "no issue tags",
                     ha="center", va="center"); axes[1].set_axis_off()

    # Marked for re-run
    rr = df["marked_for_rerun"].fillna(False).astype(bool).value_counts()
    axes[2].bar(["queued", "not queued"],
                [int(rr.get(True, 0)), int(rr.get(False, 0))],
                color=["#d9534f", "#a0a0a0"])
    axes[2].set_title("Re-run queue")

    plt.tight_layout(); plt.show()

    cols = ["subject", "session", "rating", "rating_label",
            "issue_tags", "marked_for_rerun", "n_edits", "reviewer"]
    print()
    print(df[cols].to_string(index=False))
# === QC WORKFLOW BLOCK END ===


### Reset QC (clear ratings)

QC ratings persist on disk in `<lesion>.qc.json` sidecars next to each
mask, so kernel restarts don't wipe them. The cell below lets you
deliberately clear ratings — for one stage, one subject, or all
subjects. Defaults to **dry-run** so you preview what would change
before committing.


In [ ]:
# === RESET QC BLOCK ===
# ============================================================
# Reset QC ratings (deliberate clear)
# ------------------------------------------------------------
# Configure the scope below. By default this is a DRY-RUN — change
# DRY_RUN = False to actually write changes.
# ============================================================
import sys, importlib
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import linda_qc as q
importlib.reload(q)

# ---------------- CONFIG ----------------
# Scope: leave any of these as None to widen the scope.
RESET_SUBJECT = None      # e.g. "sub-M2040" — limit to one subject
RESET_SESSION = None      # e.g. "ses-842"   — limit to one session
RESET_STAGE   = None      # one of: "skull_strip", "registration", "lesion"
                          # None = clear all stages

# Behaviour
KEEP_EDITS = True         # True  = keep the per-edit history (audit trail)
                          # False = delete the .qc.json files outright
DRY_RUN    = True         # True  = preview what would change, write nothing
                          # False = actually apply the reset
# -----------------------------------------

print("Scope:")
print(f"  subject    = {RESET_SUBJECT}")
print(f"  session    = {RESET_SESSION}")
print(f"  stage      = {RESET_STAGE}  (None = all stages)")
print(f"  keep_edits = {KEEP_EDITS}")
print(f"  dry_run    = {DRY_RUN}")
print()

affected = q.reset_qc_sidecars(
    DERIV_DIR,
    subject=RESET_SUBJECT,
    session=RESET_SESSION,
    stage=RESET_STAGE,
    keep_edits=KEEP_EDITS,
    dry_run=DRY_RUN,
)

if not affected:
    print("No matching sidecars found.")
else:
    verb = "Would touch" if DRY_RUN else ("Deleted" if not KEEP_EDITS else "Reset")
    print(f"{verb} {len(affected)} sidecar(s):")
    for p in affected:
        print(f"  {p}")
    if DRY_RUN:
        print()
        print("DRY_RUN=True — no changes written. "
              "Set DRY_RUN=False and re-run to apply.")
# === END RESET QC BLOCK ===


### Group lesion overlap map

A frequency map showing, for each MNI voxel, the proportion of subjects whose
lesion includes that voxel. This is a quick way to see where lesions cluster
across the cohort.


In [ ]:
# ============================================================
# Build a group lesion-frequency map in MNI space.
# All subject lesions are resampled into a common reference (the
# first lesion that exists), binarized at CONFIG["LESION_THRESHOLD"],
# averaged, and saved to derivatives.
# ============================================================
have_lesion = [e for e in SUBJECTS if lesion_mni_for(e).exists()]

if not have_lesion:
    print("No lesion masks available yet — nothing to summarize.")
else:
    ref_img = nib.load(str(lesion_mni_for(have_lesion[0])))
    ref_shape = ref_img.shape
    accum = np.zeros(ref_shape, dtype=np.float32)
    n_used = 0
    for entry in have_lesion:
        try:
            img = nib.load(str(lesion_mni_for(entry)))
            if img.shape != ref_shape:
                img = image.resample_to_img(
                    img, ref_img, interpolation="nearest", force_resample=True,
                    copy_header=True,
                )
            data = (img.get_fdata() > CONFIG["LESION_THRESHOLD"]).astype(np.float32)
            accum += data
            n_used += 1
        except Exception as e:
            print(f"  ⚠ {entry['subject']}: {e}")

    if n_used == 0:
        print("No usable lesions — group map not built.")
    else:
        freq = accum / n_used                                  # 0..1
        freq_img = nib.Nifti1Image(freq, ref_img.affine, ref_img.header)
        out_path = DERIV_DIR / "group_lesion_frequency.nii.gz"
        nib.save(freq_img, str(out_path))
        print(f"✓ Group lesion frequency map saved to {out_path} "
              f"(n={n_used}, max overlap = {freq.max():.2f})")

        # Static glass-brain plot for a quick overview
        try:
            glass_brain_disp = plotting.plot_glass_brain(
                freq_img, threshold=1.0 / n_used, colorbar=True,
                title=f"Lesion frequency (n={n_used})",
                cmap="hot",
            )
            plt.show()
        except Exception as e:
            print(f"  (glass-brain plot failed: {e})")

        # Interactive viewer
        try:
            from ipyniivue import NiiVue
            nv = NiiVue()
            vols = [{"path": str(out_path), "colormap": "hot", "opacity": 0.8}]
            nv.load_volumes(vols)
            from IPython.display import display as _disp
            _disp(nv)
        except Exception as e:
            print(f"  (interactive viewer failed: {e})")


### Atlas overlap

For every subject with a lesion mask, compute the overlap between the lesion and
each region of every atlas in `CONFIG["ATLASES"]`. Per-atlas results are cached
to CSV in `DERIV_DIR/lesion_atlas_overlap_<ATLAS>.csv` so the analysis only re-runs
when the source data changes.


In [ ]:
!sudo apt-get update -qq
!sudo apt-get install -y -qq ca-certificates
!sudo update-ca-certificates


In [ ]:
# ============================================================
# Compute lesion–atlas overlap for every subject and atlas.
# Results saved as one CSV per atlas in DERIV_DIR.
# ============================================================
def _atlas_arrays(atlas):
    """Pull (img, data, labels) from any nilearn-style atlas object."""
    img = atlas.maps if hasattr(atlas.maps, "get_fdata") else nib.load(atlas.maps)
    return img, img.get_fdata(), atlas.labels


def calculate_overlap(lesion_path, atlas_img, atlas_data, atlas_labels,
                      threshold=0.5):
    """Return dict[region] -> stats, plus total lesion voxel count."""
    if not Path(lesion_path).exists():
        return {}, 0
    lesion_img = nib.load(str(lesion_path))
    lesion_resampled = image.resample_to_img(
        lesion_img, atlas_img, interpolation="nearest",
        force_resample=True, copy_header=True,
    )
    lesion_binary = (lesion_resampled.get_fdata() > threshold).astype(int)
    total = int(np.sum(lesion_binary))
    if total == 0:
        return {}, 0
    overlaps = {}
    for region_idx in np.unique(atlas_data):
        if region_idx == 0:
            continue
        region_idx = int(region_idx)
        region_mask = (atlas_data == region_idx)
        n_overlap = int(np.sum(lesion_binary & region_mask))
        if n_overlap == 0:
            continue
        n_region = int(np.sum(region_mask))
        name = (atlas_labels[region_idx]
                if region_idx < len(atlas_labels) else f"Region_{region_idx}")
        overlaps[name] = {
            "lesion_in_roi_percent": 100.0 * n_overlap / total,
            "roi_coverage_percent":  100.0 * n_overlap / n_region,
            "overlap_voxels":        n_overlap,
            "total_region_voxels":   n_region,
        }
    return overlaps, total


have_lesion = [e for e in SUBJECTS if lesion_mni_for(e).exists()]
print(f"Computing overlap for {len(have_lesion)} subject(s) across "
      f"{len(ALL_ATLASES)} atlas(es)")

OVERLAP_DFS = {}                # name -> DataFrame, kept in memory
for atlas_name, atlas in ALL_ATLASES.items():
    print(f"\n--- {atlas_name} ---")
    img, data, labels = _atlas_arrays(atlas)

    # Save the atlas image once so the QC dashboard can overlay it
    atlas_path = DERIV_DIR / f"{atlas_name}_atlas.nii.gz"
    if not atlas_path.exists():
        nib.save(img, str(atlas_path))

    rows = []
    for entry in have_lesion:
        ovl, total = calculate_overlap(
            lesion_mni_for(entry), img, data, labels,
            threshold=CONFIG["LESION_THRESHOLD"],
        )
        for region, stats in ovl.items():
            rows.append({
                "subject": entry["subject"],
                "session": entry["session"] or "",
                "atlas":   atlas_name,
                "region":  region,
                **stats,
                "total_lesion_voxels": total,
            })
    df = pd.DataFrame(rows)
    OVERLAP_DFS[atlas_name] = df
    csv = DERIV_DIR / f"lesion_atlas_overlap_{atlas_name}.csv"
    df.to_csv(csv, index=False)
    print(f"  ✓ {len(df)} rows → {csv.name}")

print("\nDone.")


### Atlas overlap plots

Two views of the overlap data:

1. **Per-subject top regions** — a horizontal bar chart of the regions with the
   largest share of each subject's lesion (interactive).
2. **Cross-subject heatmap** — for the active atlas, the % of each subject's
   lesion that falls in each of the most frequently affected regions.
3. **Schaefer 7-network summary** — when Schaefer400 is in `CONFIG["ATLASES"]`,
   a stacked bar of lesion share per Yeo 7-network for every subject.


In [ ]:
# ============================================================
# Interactive atlas-overlap plots
# - Auto-updates on every widget change (no "Plot" button)
# - Works for any atlas in OVERLAP_DFS
# - Metric toggle: "lesion in ROI %"  vs  "ROI coverage %"
# - Top-N slider lets you zoom in/out
# ============================================================
import textwrap
import ipywidgets as widgets
from IPython.display import display, clear_output


METRICS = {
    "Lesion in ROI %": "lesion_in_roi_percent",
    "ROI coverage %":  "roi_coverage_percent",
}


def _wrap(label, width=38):
    """Break long atlas labels so they fit on a y-tick."""
    if isinstance(label, bytes):
        label = label.decode("utf-8", "ignore")
    return "\n".join(textwrap.wrap(str(label), width=width)) or str(label)


def _subject_label(subj, ses):
    return f"{subj} {ses}".strip() if ses else subj


def _split_label(label):
    """Reverse of _subject_label — works for labels with/without a session."""
    if " " in label:
        subj, _, ses = label.partition(" ")
        return subj, ses
    return label, ""


def _bar_chart(df, subj, ses, metric_key, metric_label, top_n):
    sel = df[(df["subject"] == subj) & (df["session"] == ses)].copy()
    if sel.empty:
        print(f"No overlap data for {subj} {ses}.")
        return
    sel = sel.sort_values(metric_key, ascending=False).head(top_n)
    # Reverse for horizontal bar (largest at top)
    sel = sel.iloc[::-1]
    labels = [_wrap(r, 38) for r in sel["region"]]
    fig, ax = plt.subplots(figsize=(9, max(2.5, 0.42 * len(sel) + 1)))
    bars = ax.barh(labels, sel[metric_key].values, color="#3b6ea8")
    for bar, value in zip(bars, sel[metric_key].values):
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
                f"{value:.1f}%", va="center", fontsize=8)
    ax.set_xlabel(metric_label)
    ax.set_title(f"{subj} {ses}  —  top {len(sel)} regions")
    ax.set_xlim(0, max(sel[metric_key].max() * 1.15, 1))
    ax.grid(axis="x", linestyle=":", alpha=0.5)
    plt.tight_layout()
    plt.show()


def _heatmap(df, metric_key, metric_label, top_n):
    if df.empty:
        print("No data."); return
    # Rank regions by *maximum* contribution across subjects so small
    # regions with a single strong overlap still show up.
    region_order = (df.groupby("region")[metric_key]
                      .max().sort_values(ascending=False).head(top_n).index)
    sub = df[df["region"].isin(region_order)].copy()
    sub["subj_ses"] = sub.apply(
        lambda r: _subject_label(r["subject"], r["session"]), axis=1,
    )
    pivot = (sub.pivot_table(index="region", columns="subj_ses",
                             values=metric_key, fill_value=0)
               .loc[region_order])
    if pivot.empty:
        print("No data after filtering."); return
    fig, ax = plt.subplots(
        figsize=(max(6, 0.4 * pivot.shape[1] + 4),
                 max(3, 0.35 * pivot.shape[0] + 2))
    )
    im = ax.imshow(pivot.values, aspect="auto", cmap="magma")
    ax.set_xticks(range(pivot.shape[1]))
    ax.set_xticklabels(pivot.columns, rotation=75, ha="right", fontsize=8)
    ax.set_yticks(range(pivot.shape[0]))
    ax.set_yticklabels([_wrap(r, 30) for r in pivot.index], fontsize=8)
    ax.set_title(f"{metric_label}  —  top {pivot.shape[0]} regions "
                 f"× {pivot.shape[1]} subject(s)")
    fig.colorbar(im, ax=ax, label=metric_label)
    plt.tight_layout()
    plt.show()


def _summary_table(df, subj, ses, metric_key, top_n):
    sel = df[(df["subject"] == subj) & (df["session"] == ses)].copy()
    if sel.empty:
        return "<em>No data for this subject.</em>"
    sel = sel.sort_values(metric_key, ascending=False).head(top_n)
    total = int(sel["total_lesion_voxels"].iloc[0])
    rows = "".join(
        f"<tr><td>{r.region}</td>"
        f"<td style='text-align:right'>{r.lesion_in_roi_percent:.1f}%</td>"
        f"<td style='text-align:right'>{r.roi_coverage_percent:.1f}%</td>"
        f"<td style='text-align:right'>{int(r.overlap_voxels)}</td></tr>"
        for r in sel.itertuples()
    )
    return (
        f"<p><b>Total lesion volume:</b> {total} voxels &nbsp; "
        f"<b>Regions overlapped:</b> {len(sel)}</p>"
        f"<table style='border-collapse:collapse; font-size:12px'>"
        f"<tr style='background:#2c3e50; color:white'>"
        f"<th style='padding:4px 8px; text-align:left'>Region</th>"
        f"<th style='padding:4px 8px'>Lesion%</th>"
        f"<th style='padding:4px 8px'>ROI%</th>"
        f"<th style='padding:4px 8px'>Voxels</th></tr>"
        f"{rows}</table>"
    )


if not OVERLAP_DFS:
    print("Run the atlas-overlap cell first — no data to plot.")
else:
    # ---- Widgets ----------------------------------------------
    atlas_dd = widgets.Dropdown(
        options=list(OVERLAP_DFS.keys()),
        value=(CONFIG["DEFAULT_ATLAS"] if CONFIG["DEFAULT_ATLAS"] in OVERLAP_DFS
               else list(OVERLAP_DFS.keys())[0]),
        description="Atlas:",
        layout=widgets.Layout(width="240px"),
        style={"description_width": "initial"},
    )
    subj_dd = widgets.Dropdown(
        options=[], description="Subject:",
        layout=widgets.Layout(width="300px"),
        style={"description_width": "initial"},
    )
    metric_tb = widgets.ToggleButtons(
        options=list(METRICS.keys()), description="Metric:",
        style={"description_width": "initial"},
    )
    top_n_sl = widgets.IntSlider(
        value=20, min=5, max=50, step=1, description="Top N:",
        continuous_update=False,
        layout=widgets.Layout(width="320px"),
        style={"description_width": "initial"},
    )

    def _refresh_subjects(*_):
        df = OVERLAP_DFS[atlas_dd.value]
        pairs = sorted(set(zip(df["subject"], df["session"].fillna(""))))
        labels = [_subject_label(s, ses) for s, ses in pairs]
        subj_dd.options = labels
        if labels:
            subj_dd.value = labels[0]

    atlas_dd.observe(_refresh_subjects, names="value")
    _refresh_subjects()

    bar_out   = widgets.Output()
    table_out = widgets.Output()
    heat_out  = widgets.Output()

    def _redraw(*_):
        from IPython.display import HTML
        if not OVERLAP_DFS:
            return
        df = OVERLAP_DFS[atlas_dd.value]
        metric_label = metric_tb.value
        metric_key   = METRICS[metric_label]
        subj, ses    = _split_label(subj_dd.value) if subj_dd.value else ("", "")
        top_n        = top_n_sl.value

        with bar_out:
            clear_output(wait=True)
            _bar_chart(df, subj, ses, metric_key, metric_label, top_n)
        with table_out:
            clear_output(wait=True)
            display(HTML(_summary_table(df, subj, ses, metric_key, top_n)))
        with heat_out:
            clear_output(wait=True)
            _heatmap(df, metric_key, metric_label, top_n)

    for w in (atlas_dd, subj_dd, metric_tb, top_n_sl):
        w.observe(_redraw, names="value")

    controls = widgets.VBox([
        widgets.HBox([atlas_dd, subj_dd]),
        widgets.HBox([metric_tb, top_n_sl]),
    ])

    tabs = widgets.Tab(children=[
        widgets.VBox([bar_out, table_out]),
        heat_out,
    ])
    tabs.set_title(0, "Per-subject")
    tabs.set_title(1, "Cross-subject heatmap")

    display(controls, tabs)
    _redraw()


In [ ]:
# ============================================================
# Schaefer 7-network summary (only runs if Schaefer400 was processed)
# ============================================================
NET_NAMES = ["Vis", "SomMot", "DorsAttn", "SalVentAttn", "Limbic", "Cont", "Default"]


def _net_for(label):
    """Map a Schaefer region label to one of the 7 Yeo networks."""
    if isinstance(label, bytes):
        label = label.decode("utf-8", "ignore")
    for n in NET_NAMES:
        if n in label:
            return n
    return "Other"


if "Schaefer400" not in OVERLAP_DFS or OVERLAP_DFS["Schaefer400"].empty:
    print("Schaefer400 overlap not available — skipping network summary.")
else:
    df = OVERLAP_DFS["Schaefer400"].copy()
    df["network"] = df["region"].astype(str).map(_net_for)
    summary = (df.groupby(["subject", "session", "network"])
                 ["lesion_in_roi_percent"].sum().unstack(fill_value=0))
    cols = [n for n in NET_NAMES if n in summary.columns]
    if "Other" in summary.columns:
        cols.append("Other")
    summary = summary[cols]
    fig, ax = plt.subplots(figsize=(max(6, 0.5 * len(summary) + 3), 5))
    bottom = np.zeros(len(summary))
    x = np.arange(len(summary))
    for col in summary.columns:
        ax.bar(x, summary[col].values, bottom=bottom, label=col)
        bottom += summary[col].values
    ax.set_xticks(x)
    ax.set_xticklabels([f"{s} {ses}".strip() for s, ses in summary.index],
                       rotation=90, fontsize=8)
    ax.set_ylabel("Sum of lesion-in-ROI % (Schaefer regions)")
    ax.set_title("Lesion distribution across Yeo 7 networks")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
    plt.tight_layout()
    plt.show()


### Per-subject report (interactive + PDF)

Pick a subject and an atlas to see the full regional breakdown alongside the
lesion volume in MNI space. The same data can be exported as PDF reports — set
`GENERATE_ALL = True` to write one PDF per subject for the active atlas.


In [ ]:
# ============================================================
# Interactive per-subject report (table + viewer)
# ============================================================
from ipyniivue import NiiVue
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output


def _report_html(df, subj, ses, atlas_name):
    sel = df[(df["subject"] == subj) & (df["session"] == (ses or ""))].copy()
    if sel.empty:
        return f"<p>No data for {subj} {ses}.</p>"
    sel = sel.sort_values("lesion_in_roi_percent", ascending=False)
    total = int(sel["total_lesion_voxels"].iloc[0])
    rows = "".join(
        f"<tr><td>{r.region}</td>"
        f"<td style='text-align:right'>{r.lesion_in_roi_percent:.1f}%</td>"
        f"<td style='text-align:right'>{r.roi_coverage_percent:.1f}%</td>"
        f"<td style='text-align:right'>{int(r.overlap_voxels)}</td></tr>"
        for r in sel.itertuples()
    )
    return f"""
    <h3>{subj} {ses or ''} — {atlas_name}</h3>
    <p><b>Total lesion volume:</b> {total} voxels &nbsp;
       <b>Regions affected:</b> {len(sel)}</p>
    <table style='border-collapse:collapse; font-size:12px'>
      <tr style='background:#2c3e50; color:white'>
        <th style='padding:6px 10px; text-align:left'>Region</th>
        <th style='padding:6px 10px'>Lesion in ROI</th>
        <th style='padding:6px 10px'>ROI coverage</th>
        <th style='padding:6px 10px'>Voxels</th>
      </tr>{rows}
    </table>"""


if not OVERLAP_DFS:
    print("Run the atlas-overlap cell first.")
else:
    atlas_dd = widgets.Dropdown(
        options=list(OVERLAP_DFS.keys()), description="Atlas:",
        value=(CONFIG["DEFAULT_ATLAS"] if CONFIG["DEFAULT_ATLAS"] in OVERLAP_DFS
               else list(OVERLAP_DFS.keys())[0]),
    )
    subj_dd = widgets.Dropdown(description="Subject:", options=[])

    def _refresh_subjects(*_):
        df = OVERLAP_DFS[atlas_dd.value]
        pairs = sorted(set(zip(df["subject"], df["session"])))
        subj_dd.options = [f"{s} {ses}".strip() for s, ses in pairs]
        if subj_dd.options:
            subj_dd.value = subj_dd.options[0]

    _refresh_subjects()
    atlas_dd.observe(_refresh_subjects, names="value")

    report_out = widgets.Output()
    viz_out    = widgets.Output()

    def _show_report(*_):
        with report_out:
            clear_output()
            df = OVERLAP_DFS[atlas_dd.value]
            label = subj_dd.value
            subj, _, ses = label.partition(" ")
            display(HTML(_report_html(df, subj, ses, atlas_dd.value)))
        with viz_out:
            clear_output()
            label = subj_dd.value
            subj, _, ses = label.partition(" ")
            entry = next((e for e in SUBJECTS
                          if e["subject"] == subj
                          and (e["session"] or "") == ses), None)
            if entry is None:
                return
            d = deriv_path_for(entry)
            atlas_path = DERIV_DIR / f"{atlas_dd.value}_atlas.nii.gz"
            vols = []
            if (d / "Subject_in_MNI.nii.gz").exists():
                vols.append({"path": str(d / "Subject_in_MNI.nii.gz")})
            if atlas_path.exists():
                vols.append({"path": str(atlas_path),
                             "colormap": "viridis", "opacity": 0.4})
            if (d / "Lesion_in_MNI.nii.gz").exists():
                vols.append({"path": str(d / "Lesion_in_MNI.nii.gz"),
                             "colormap": "red", "opacity": 0.6})
            nv = NiiVue()
            nv.load_volumes(vols)
            display(nv)

    atlas_dd.observe(_show_report, names="value")
    subj_dd.observe(_show_report, names="value")

    display(HTML("<h3>Interactive lesion report</h3>"))
    display(widgets.HBox([atlas_dd, subj_dd]))
    display(viz_out)
    display(report_out)
    _show_report()


In [ ]:
# ============================================================
# PDF report generator — uses the in-memory OVERLAP_DFS
# ============================================================
from matplotlib.backends.backend_pdf import PdfPages

GENERATE_ALL = False                                  # set True for batch
PDF_ATLAS    = CONFIG["DEFAULT_ATLAS"]
PDF_SUBJECT  = None                                   # None = first available
PDF_SESSION  = None                                   # None = first available

REPORTS_DIR = DERIV_DIR / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)


def write_pdf(subj, ses, df, atlas_name):
    sel = df[(df["subject"] == subj) & (df["session"] == (ses or ""))].copy()
    if sel.empty:
        print(f"  no data for {subj} {ses} ({atlas_name})"); return None
    sel = sel.sort_values("lesion_in_roi_percent", ascending=False)
    total = int(sel["total_lesion_voxels"].iloc[0])
    fname = REPORTS_DIR / f"{subj}_{ses or 'no-ses'}_{atlas_name}.pdf"

    with PdfPages(str(fname)) as pdf:
        # Page 1 — summary + bar chart
        fig = plt.figure(figsize=(8.5, 11))
        plt.suptitle(f"Lesion report  —  {subj} {ses or ''}  ({atlas_name})",
                     fontsize=14, fontweight="bold")
        ax_meta = fig.add_axes([0.08, 0.78, 0.84, 0.12]); ax_meta.axis("off")
        ax_meta.text(0, 0.6,
            f"Date: {datetime.now():%Y-%m-%d %H:%M}\n"
            f"Total lesion volume: {total} voxels\n"
            f"Regions affected: {len(sel)}",
            family="monospace", fontsize=10)
        ax_bar = fig.add_axes([0.08, 0.08, 0.84, 0.62])
        top = sel.head(20)
        ax_bar.barh(top["region"][::-1], top["lesion_in_roi_percent"][::-1])
        ax_bar.set_xlabel("% of lesion in ROI")
        ax_bar.set_title(f"Top {len(top)} affected regions")
        pdf.savefig(fig); plt.close(fig)

        # Subsequent pages — full table
        rows_per_page = 35
        for start in range(0, len(sel), rows_per_page):
            chunk = sel.iloc[start:start + rows_per_page]
            fig = plt.figure(figsize=(8.5, 11))
            plt.suptitle(f"All regions  —  rows {start + 1}-{start + len(chunk)} "
                         f"of {len(sel)}", fontsize=11)
            ax = fig.add_axes([0.05, 0.05, 0.9, 0.85]); ax.axis("off")
            text = (f"{'Rank':<5s}{'Region':<42s}"
                    f"{'Lesion%':>10s}{'ROI%':>8s}{'vox':>8s}\n"
                    + "-" * 75 + "\n")
            for i, row in enumerate(chunk.itertuples(), start + 1):
                text += (f"{i:<5d}{row.region[:40]:<42s}"
                         f"{row.lesion_in_roi_percent:>9.1f}%"
                         f"{row.roi_coverage_percent:>7.1f}%"
                         f"{int(row.overlap_voxels):>8d}\n")
            ax.text(0, 1, text, family="monospace", fontsize=8,
                    verticalalignment="top")
            pdf.savefig(fig); plt.close(fig)
    print(f"  ✓ {fname}")
    return fname


df = OVERLAP_DFS.get(PDF_ATLAS)
if df is None or df.empty:
    print(f"No data for atlas '{PDF_ATLAS}'.")
else:
    targets = (sorted(set(zip(df["subject"], df["session"])))
               if GENERATE_ALL
               else [(PDF_SUBJECT or df["subject"].iloc[0],
                      PDF_SESSION or df["session"].iloc[0])])
    print(f"Writing {len(targets)} PDF(s) to {REPORTS_DIR}")
    for s, sess in targets:
        write_pdf(s, sess, df, PDF_ATLAS)


## Summary

A snapshot of how the run went — what was discovered, segmented, analysed, and
produced. Useful when re-running on a new dataset.


In [ ]:
# ============================================================
# Final summary
# ============================================================
n_total      = len(SUBJECTS)
n_lesion     = sum(1 for e in SUBJECTS if lesion_mni_for(e).exists())
atlas_csvs   = list(DERIV_DIR.glob("lesion_atlas_overlap_*.csv"))
group_map    = DERIV_DIR / "group_lesion_frequency.nii.gz"
report_files = list((DERIV_DIR / "reports").glob("*.pdf")) if (DERIV_DIR / "reports").exists() else []

print("=" * 60)
print("LINDA notebook run summary")
print("=" * 60)
print(f"Project dir         : {PROJECT_DIR}")
print(f"Dataset source      : {CONFIG['DATASET_SOURCE']}")
print(f"Subjects discovered : {n_total}")
print(f"With lesion masks   : {n_lesion}")
print(f"Atlases processed   : {len(atlas_csvs)}  "
      f"({', '.join(c.stem.replace('lesion_atlas_overlap_', '') for c in atlas_csvs)})")
print(f"Group lesion map    : {'yes' if group_map.exists() else 'no'}")
print(f"PDF reports written : {len(report_files)}")
print()
print(f"All outputs are under: {DERIV_DIR}")
